In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 51.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [3]:
import os, gc, copy, math, warnings, logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator

from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference
from fairlearn.adversarial import AdversarialFairnessClassifier

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR   = './cache'
RESULTS_DIR = '/kaggle/working'
for d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def floor2(x): return math.floor(abs(float(x)) * 100) / 100
def r4(x):     return round(abs(float(x)), 4)

DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}

ACC_FLOORS = {
    "adult": 0.75, "compas": 0.62, "german": 0.62,
    "bank": 0.77, "lawschool": 0.88, "hospital": 0.54,
}

EPOCH_CFG = {
    "german":    dict(total=80,  warmup=20, patience=10),
    "adult":     dict(total=70,  warmup=15, patience=10),
    "compas":    dict(total=70,  warmup=15, patience=10),
    "bank":      dict(total=60,  warmup=12, patience=10),
    "lawschool": dict(total=50,  warmup=12, patience=10),
    "hospital":  dict(total=60,  warmup=12, patience=10),
}

STAGE_RECORDS:       Dict[str, Dict] = {}
MEDIATOR_SCORES_ALL: Dict[str, Dict] = {}
TRAINING_CURVES:     Dict[str, Dict] = {}
ABLATION_RECORDS:    Dict[str, Dict] = {}


def to_dense(X):
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_num(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s, q=5):
    s = clean_num(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except:
        return pd.Series(0, index=s.index, dtype=int)

def num_pipe():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])

def cat_pipe():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

def _enc_bbn_objects(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].astype('category').cat.codes.astype(int)
    return df


def load_adult(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/adult.csv',
               seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_adult.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] Adult"); return joblib.load(cache)
    cols = ['age','workclass','fnlwgt','education','education-num','marital-status',
            'occupation','relationship','race','sex','capital-gain','capital-loss',
            'hours-per-week','native-country','income']
    df = None
    for sep in [', ', ',']:
        try:
            peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
            if peek.shape[1] == 15:
                first = str(peek.iloc[0, 0]).strip()
                df = (pd.read_csv(csv_path, sep=sep, names=cols, header=None)
                      if first.lstrip('-').isdigit()
                      else pd.read_csv(csv_path, sep=sep, header=0))
                df.columns = cols; break
        except:
            continue
    if df is None:
        raise ValueError("Cannot parse Adult CSV")
    for c in df.select_dtypes('object').columns:
        df[c] = df[c].astype(str).str.strip()
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
    df['y']        = df['income'].str.contains('>50K', na=False).astype(int)
    df['sex_bin']  = (df['sex']  == 'Male').astype(int)
    df['race_bin'] = (df['race'] == 'White').astype(int)
    num_c = ['age','education-num','capital-gain','capital-loss','hours-per-week']
    cat_c = ['workclass','education','marital-status','occupation','relationship','native-country']
    for c in num_c: df[c] = clean_num(df[c])
    for c in ['capital-gain','capital-loss']: df[c] = df[c].clip(upper=df[c].quantile(0.99))
    X = df.drop(columns=['income','y','sex_bin','race_bin','sex','race'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,ss_tr,ss_te,sr_tr,sr_te = train_test_split(
        X, y, df['sex_bin'].values, df['race_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.drop(columns=['income']).copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['sex','race']:
        if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn['y'] = bbn['y'].astype(int); bbn = _enc_bbn_objects(bbn)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             sens_race_train=sr_tr, sens_race_test=sr_te)
    if use_cache: joblib.dump(r, cache)
    return r


def load_compas(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/compas-scores-two-years.csv',
                seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_compas.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] COMPAS"); return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'].between(-30, 30)) & (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
         'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
         'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    df['race_bin'] = (df['race'] == 'African-American').astype(int)
    df['sex_bin']  = (df['sex']  == 'Male').astype(int)
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    num_c = ['age','priors_count','days_b_screening_arrest','decile_score',
             'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    cat_c = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    for c in num_c: df[c] = clean_num(df[c])
    X = df.drop(columns=['is_recid','two_year_recid','race_bin','sex_bin'])
    y = df['two_year_recid'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te = train_test_split(
        X, y, df['race_bin'], df['sex_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['race','sex']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
             sens_sex_train=ss_tr.reset_index(drop=True),  sens_sex_test=ss_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache)
    return r


def load_german(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/german.data',
                seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_german.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] German"); return joblib.load(cache)
    cols = ["status","duration","credit_history","purpose","amount","savings","employment",
            "installment_rate","personal_status_sex","other_debtors","residence","property","age",
            "other_installment_plans","housing","number_credits","job","people_liable",
            "telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=' ', names=cols)
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex']     = df['personal_status_sex'].map(sex_map)
    df['sex_bin'] = (df['sex'] == 'male').astype(int)
    df['age_bin'] = (df['age'] >= 25).astype(int)
    df['y']       = df['credit'].map({1:1, 2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    num_c = ["duration","amount","installment_rate","residence","number_credits","people_liable"]
    cat_c = [c for c in df.columns if c not in num_c + ['sex_bin','age_bin','y']]
    for c in num_c: df[c] = clean_num(df[c])
    X = df.drop(columns=['y'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,sa_tr,sa_te,ss_tr,ss_te = train_test_split(
        X, y, df['age_bin'].values, df['sex_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['sex_bin','age_bin']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_age_train=sa_tr, sens_age_test=sa_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te)
    if use_cache: joblib.dump(r, cache)
    return r


def load_bank(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bank-full.csv',
              seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_bank.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] Bank"); return joblib.load(cache)
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path)
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    tc = 'y' if 'y' in df.columns else ('deposit' if 'deposit' in df.columns else df.columns[-1])
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y']           = np.where(df[tc].isin(['yes','y','true','1']), 1, 0)
    df['marital_bin'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
    df['job_bin']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)
    cat_c = [c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
    num_c = [c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
    for c in num_c: df[c] = clean_num(df[c])
    for c in ['balance','duration','pdays','previous']:
        if c in df.columns: df[c] = df[c].clip(upper=df[c].quantile(0.99))
    X = df.drop(columns=['y','marital_bin','job_bin'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,sm_tr,sm_te,sj_tr,sj_te = train_test_split(
        X, y, df['marital_bin'], df['job_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['marital','job']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_marital_train=sm_tr.reset_index(drop=True), sens_marital_test=sm_te.reset_index(drop=True),
             sens_job_train=sj_tr.reset_index(drop=True),     sens_job_test=sj_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache)
    return r


def load_lawschool(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bar_pass_prediction.csv',
                   use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_lawschool.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] LawSchool"); return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=['pass_bar','race','sex'])
    for c in df.select_dtypes(include=[np.number]).columns: df[c] = clean_num(df[c])
    mc = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race'] == mc).astype(int)
    gm = {v:i for i,v in enumerate(sorted(df['sex'].unique()))}
    df['gender_bin'] = df['sex'].map(gm)
    num_c = [c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa']
             if c in df.columns and df[c].nunique() > 1]
    cat_c = [c for c in ['tier','cluster','fulltime','parttime'] if c in df.columns]
    X = df[num_c + cat_c + ['race','sex']]
    y = df['pass_bar'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=SEED)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c+['race','sex'])])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(df[c], 5)
    for c in cat_c+['race','sex']:
        if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    if 'pass_bar' in bbn.columns and bbn['pass_bar'].dtype == object:
        bbn['pass_bar'] = bbn['pass_bar'].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True),   sens_race_test=sr_te.reset_index(drop=True),
             sens_gender_train=sg_tr.reset_index(drop=True), sens_gender_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache)
    return r


def load_hospital(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/diabetes_hospital_fairlearn.csv',
                  seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_hospital.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] Hospital"); return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ['max_glu_serum','A1Cresult','readmitted.1'] if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
    age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
               "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
               "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age']        = df['age'].replace(age_map).astype(float)
    df['y']          = df['readmit_binary'].astype(int)
    mc               = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race'] == mc).astype(int)
    df['gender_bin'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
    cat_c = ['discharge_disposition_id','admission_source_id','medical_specialty',
             'primary_diagnosis','insulin','change','diabetesMed']
    num_c = ['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
             'number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid']
    for c in num_c: df[c] = clean_num(df[c])
    X = df.drop(columns=['readmit_binary','y','race_bin','gender_bin'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['race','gender']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
             sens_sex_train=sg_tr.reset_index(drop=True),  sens_sex_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache)
    return r


def _compute_mi(a, b):
    return float(mutual_info_score(a.astype(int), b.astype(int)))


@dataclass
class MediatorBiasScores:
    sens_col:    str
    label_col:   str
    scores:      Dict[str, float] = field(default_factory=dict)
    sens_mi:     float = 0.0
    feature_cols: List[str] = field(default_factory=list)

    def fit(self, bbn_df: pd.DataFrame):
        df = bbn_df.copy()
        if self.sens_col not in df.columns or self.label_col not in df.columns:
            return self
        s = df[self.sens_col].values.astype(int)
        y = df[self.label_col].values.astype(int)
        self.sens_mi = _compute_mi(s, y)
        candidates = [c for c in df.columns if c not in [self.sens_col, self.label_col]]
        self.feature_cols = candidates
        for feat in candidates:
            f = df[feat].values.astype(int)
            mi_sf = _compute_mi(s, f)
            mi_fy = _compute_mi(f, y)
            dep = abs(np.corrcoef(s.astype(float), f.astype(float))[0, 1]) if f.std() > 0 else 0.0
            self.scores[feat] = float(0.45 * mi_sf + 0.40 * mi_fy + 0.15 * dep)
        if self.scores:
            mx = max(self.scores.values()) + 1e-9
            self.scores = {k: v / mx for k, v in self.scores.items()}
        return self

    def top_mediators(self, k: int = 10, threshold: float = 0.1) -> List[Tuple[str, float]]:
        ranked = sorted(self.scores.items(), key=lambda x: x[1], reverse=True)
        return [(f, s) for f, s in ranked if s >= threshold][:k]


@dataclass
class BBNPathAnalyzer:
    sens_col:    str
    label_col:   str
    bias_scores: MediatorBiasScores = field(default=None)
    _pathway_weights: Dict[str, float] = field(default_factory=dict, repr=False)
    BBN_SUBSAMPLE = 500

    def fit(self, bbn_df: pd.DataFrame):
        df = bbn_df.copy()
        self.bias_scores = MediatorBiasScores(sens_col=self.sens_col, label_col=self.label_col)
        self.bias_scores.fit(df)
        top = self.bias_scores.top_mediators(k=6, threshold=0.05)
        self._pathway_weights = {f: s for f, s in top}
        top_feats = [f for f, _ in top]

        if len(df) > self.BBN_SUBSAMPLE:
            df_bbn = df.sample(self.BBN_SUBSAMPLE, random_state=42)
        else:
            df_bbn = df

        if self.sens_col not in df_bbn.columns:
            return self
        edges = [(self.sens_col, n) for n in top_feats if n in df_bbn.columns]
        if self.label_col in df_bbn.columns and self.label_col not in top_feats:
            edges += [(self.sens_col, self.label_col)]
        if not edges:
            return self
        try:
            model = DiscreteBayesianNetwork(edges)
            model.fit(df_bbn, estimator=BayesianEstimator,
                      prior_type='BDeu', equivalent_sample_size=5)
        except:
            pass
        return self

    def get_pathway_weights(self) -> Dict[str, float]:
        return self._pathway_weights

    def get_sens_mi(self) -> float:
        return self.bias_scores.sens_mi if self.bias_scores else 0.0

    def get_feature_bias_scores(self) -> Dict[str, float]:
        return self.bias_scores.scores if self.bias_scores else {}


class GradRevFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        if isinstance(ctx.alpha, torch.Tensor):
            alpha = ctx.alpha.view(-1, 1)
        else:
            alpha = ctx.alpha
        return -alpha * grad_output, None


class FairnessEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, z_dim=128, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.b1   = nn.Sequential(nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, hidden))
        self.b2   = nn.Sequential(nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, hidden))
        self.out  = nn.Sequential(nn.LayerNorm(hidden), nn.GELU(), nn.Linear(hidden, z_dim), nn.LayerNorm(z_dim))

    def forward(self, x):
        h = self.proj(x); h = h + self.b1(h); h = h + self.b2(h); return self.out(h)


class TaskHead(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.fc = nn.Linear(z_dim, 1)

    def forward(self, z):
        return self.fc(z).squeeze(-1)


class BBNAdversary(nn.Module):
    def __init__(self, z_dim=128, hidden=64):
        super().__init__()
        self.alpha = 1.0
        self.net   = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, 2)
        )

    def forward(self, z, alpha=None):
        if alpha is None:
            alpha = self.alpha
        z = GradRevFn.apply(z, alpha)
        return self.net(z)

    def set_alpha(self, a: float):
        self.alpha = float(a)


def fair_loss_eo(logit: torch.Tensor, y: torch.Tensor, s: torch.Tensor,
                 margin: float = 0.005) -> torch.Tensor:
    p = torch.sigmoid(logit / 2.0)
    tprs, fprs = [], []
    for g in (0, 1):
        pm = (s == g) & (y == 1)
        nm = (s == g) & (y == 0)
        tprs.append(p[pm].mean() if pm.sum() > 1 else torch.tensor(0.5, device=logit.device))
        fprs.append(p[nm].mean() if nm.sum() > 1 else torch.tensor(0.5, device=logit.device))
    return F.relu(torch.max(torch.abs(tprs[0] - tprs[1]),
                             torch.abs(fprs[0] - fprs[1])) - margin)


def fair_loss_dp(logit: torch.Tensor, s: torch.Tensor,
                 margin: float = 0.005) -> torch.Tensor:
    p = torch.sigmoid(logit / 2.0)
    p0 = p[s == 0].mean() if (s == 0).sum() > 1 else torch.tensor(0.5, device=logit.device)
    p1 = p[s == 1].mean() if (s == 1).sum() > 1 else torch.tensor(0.5, device=logit.device)
    return F.relu(torch.abs(p0 - p1) - margin)


def _roc_points(proba: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    thresholds = np.unique(proba)
    tprs, fprs = [], []
    pos = y.sum(); neg = len(y) - pos
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tp   = ((pred == 1) & (y == 1)).sum()
        fp   = ((pred == 1) & (y == 0)).sum()
        tprs.append(tp / pos if pos > 0 else 0.0)
        fprs.append(fp / neg if neg > 0 else 0.0)
    fprs_arr = np.array(fprs); tprs_arr = np.array(tprs)
    sort_idx = np.argsort(fprs_arr)
    return fprs_arr[sort_idx], tprs_arr[sort_idx]


def _convex_hull_roc(fprs: np.ndarray, tprs: np.ndarray, n_pts: int = 200) -> Tuple[np.ndarray, np.ndarray]:
    pts = np.stack([fprs, tprs], axis=1)
    pts = np.vstack([[0.0, 0.0], pts, [1.0, 1.0]])
    pts = pts[np.argsort(pts[:, 0])]
    hull_fprs = [pts[0, 0]]; hull_tprs = [pts[0, 1]]
    for i in range(1, len(pts)):
        while len(hull_fprs) >= 2:
            dx1 = hull_fprs[-1] - hull_fprs[-2]; dy1 = hull_tprs[-1] - hull_tprs[-2]
            dx2 = pts[i, 0] - hull_fprs[-2];     dy2 = pts[i, 1] - hull_tprs[-2]
            if dx1 * dy2 <= dy1 * dx2:
                hull_fprs.pop(); hull_tprs.pop()
            else:
                break
        hull_fprs.append(pts[i, 0]); hull_tprs.append(pts[i, 1])
    hull_fprs = np.array(hull_fprs); hull_tprs = np.array(hull_tprs)
    interp_fprs = np.linspace(0, 1, n_pts)
    interp_tprs = np.interp(interp_fprs, hull_fprs, hull_tprs)
    return interp_fprs, interp_tprs


def calibrate_proba(proba_tr: np.ndarray, y_tr: np.ndarray,
                     proba_te: np.ndarray) -> np.ndarray:
    try:
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(proba_tr, y_tr)
        return iso.predict(proba_te).astype(np.float64)
    except:
        return proba_te


def hardt_postprocess(
    proba: np.ndarray,
    y: np.ndarray,
    s: np.ndarray,
    acc_floor: float,
    target: str = 'eo',
    random_state: int = SEED,
    proba_cal_src: Tuple[np.ndarray, np.ndarray] = None,
    n_roc_pts: int = 200,
) -> Tuple[np.ndarray, float, float]:
    rng = np.random.default_rng(random_state)
    s   = s.astype(int)

    if proba_cal_src is not None:
        proba_tr_cal, y_tr_cal = proba_cal_src
        proba = calibrate_proba(proba_tr_cal, y_tr_cal, proba)

    roc = {}
    for g in (0, 1):
        m = s == g
        if m.sum() < 10:
            roc[g] = (np.array([0.0, 1.0]), np.array([0.0, 1.0]))
        else:
            raw_fprs, raw_tprs = _roc_points(proba[m], y[m])
            hull_fprs, hull_tprs = _convex_hull_roc(raw_fprs, raw_tprs, n_pts=n_roc_pts)
            roc[g] = (hull_fprs, hull_tprs)

    pos0 = (y[s==0] == 1).sum(); neg0 = (y[s==0] == 0).sum()
    pos1 = (y[s==1] == 1).sum(); neg1 = (y[s==1] == 0).sum()
    n0   = (s==0).sum();          n1   = (s==1).sum()
    n    = len(y)

    def pred_from_roc(g, fpr_val, tpr_val):
        m   = s == g
        pg  = proba[m]; yg = y[m]
        pos = yg == 1; neg_m = yg == 0
        pred_g = np.zeros(m.sum(), dtype=np.float32)
        if pos.sum() > 0:
            n_tp = int(round(tpr_val * pos.sum()))
            top_pos = np.argsort(-pg[pos])[:n_tp]
            idx_pos = np.where(pos)[0][top_pos]
            pred_g[idx_pos] = 1.0
        if neg_m.sum() > 0:
            n_fp = int(round(fpr_val * neg_m.sum()))
            top_neg = np.argsort(-pg[neg_m])[:n_fp]
            idx_neg = np.where(neg_m)[0][top_neg]
            pred_g[idx_neg] = 1.0
        return pred_g

    best_metric = 1.0; best_acc = 0.0; best_pred = None

    fpr0, tpr0 = roc[0]
    fpr1, tpr1 = roc[1]

    floor_guard = acc_floor * 0.95

    for i in range(len(fpr0)):
        for j in range(len(fpr1)):
            f0, t0 = fpr0[i], tpr0[i]
            f1, t1 = fpr1[j], tpr1[j]

            if target == 'eo':
                metric = max(abs(t0 - t1), abs(f0 - f1))
            else:
                pr0 = t0 * (pos0/n0) + f0 * (neg0/n0) if n0 > 0 else 0.0
                pr1 = t1 * (pos1/n1) + f1 * (neg1/n1) if n1 > 0 else 0.0
                metric = abs(pr0 - pr1)

            acc = ((t0 * pos0 + (1 - f0) * neg0) / n0 * n0 +
                   (t1 * pos1 + (1 - f1) * neg1) / n1 * n1) / n if n > 0 else 0.0

            if acc < floor_guard:
                continue
            if metric < best_metric or (abs(metric - best_metric) < 1e-6 and acc > best_acc):
                best_metric = metric
                best_acc    = acc
                best_pred   = (f0, t0, f1, t1)

    if best_pred is None:
        tqdm.write(f"    [Hardt] no feasible point found, using 0.5 threshold")
        pred = (proba >= 0.5).astype(int)
        try:
            if target == 'eo':
                m = r4(equalized_odds_difference(y, pred, sensitive_features=s))
            else:
                m = r4(demographic_parity_difference(y, pred, sensitive_features=s))
        except:
            m = 1.0
        return pred, m, float((pred == y).mean())

    f0, t0, f1, t1 = best_pred
    pred = np.zeros(n, dtype=int)
    pred[s == 0] = pred_from_roc(0, f0, t0).astype(int)
    pred[s == 1] = pred_from_roc(1, f1, t1).astype(int)

    try:
        if target == 'eo':
            final_metric = r4(equalized_odds_difference(y, pred, sensitive_features=s))
        else:
            final_metric = r4(demographic_parity_difference(y, pred, sensitive_features=s))
    except:
        final_metric = r4(best_metric)

    final_acc = float((pred == y).mean())
    return pred, final_metric, final_acc


@torch.no_grad()
def get_proba(enc, task_hd, X):
    enc.eval(); task_hd.eval()
    dev = next(enc.parameters()).device
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)
    return torch.sigmoid(task_hd(enc(X.to(dev)))).cpu().numpy()


def train_lcfr(dataset_name, data, bbn_analyzer: BBNPathAnalyzer,
               primary_sens_train, primary_sens_test, baseline_acc,
               baseline_eo: float = None, baseline_dp: float = None,
               ablation_mode: str = 'full'):

    cfg = EPOCH_CFG.get(dataset_name, dict(total=70, warmup=15, patience=10))
    total_ep = cfg['total']
    warmup_ep = cfg['warmup']
    patience = cfg['patience']
    acc_floor = ACC_FLOORS.get(dataset_name, baseline_acc * 0.90)
    batch = 2048

    use_bbn = ablation_mode not in ('no_bbn',)
    use_adv = ablation_mode not in ('no_adv',)
    use_post = ablation_mode not in ('no_post',)

    X_tr = torch.tensor(data['X_train_nn'], dtype=torch.float32)
    y_tr = torch.tensor(data['y_train'], dtype=torch.float32)
    s_tr = torch.tensor(primary_sens_train, dtype=torch.long)

    X_te = torch.tensor(data['X_test_nn'], dtype=torch.float32)
    y_np = data['y_test']
    s_np = primary_sens_test

    # ---------- BBN mediator preparation (if used) ----------
    mediator_heads = None
    mediator_dims = None
    mediator_values_train = None

    if use_bbn and ablation_mode == 'full':
        pw = bbn_analyzer.get_pathway_weights()
        top_mediators = sorted(pw.items(), key=lambda x: x[1], reverse=True)[:5]
        if top_mediators:
            bbn_tr_df = data.get('bbn_train_df')
            if bbn_tr_df is not None:
                med_feats = [f for f, _ in top_mediators if f in bbn_tr_df.columns]
                if med_feats:
                    med_vals = []
                    for feat in med_feats:
                        vals = bbn_tr_df[feat].astype(int).values
                        med_vals.append(vals)
                    mediator_values_train = torch.tensor(np.stack(med_vals, axis=1), dtype=torch.long)
                    mediator_dims = [len(np.unique(med_vals[i])) for i in range(len(med_feats))]
                    mediator_heads = nn.ModuleDict({
                        f"med_{i}": nn.Sequential(
                            nn.Linear(128, 64),
                            nn.ReLU(),
                            nn.Linear(64, mediator_dims[i])
                        ).to(DEVICE) for i in range(len(med_feats))
                    })
                    tqdm.write(f"    [BBN] Activating mediator adversaries for: {med_feats}")

    # Build dataset with or without mediator values
    if mediator_values_train is not None:
        class MediatorDataset(torch.utils.data.Dataset):
            def __init__(self, X, y, s, med):
                self.X = X
                self.y = y
                self.s = s
                self.med = med
            def __len__(self): return len(self.X)
            def __getitem__(self, idx):
                return self.X[idx], self.y[idx], self.s[idx], self.med[idx]
        ds = MediatorDataset(X_tr, y_tr, s_tr, mediator_values_train)
        loader = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    else:
        ds = TensorDataset(X_tr, y_tr, s_tr)
        loader = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())

    # ---------- Models ----------
    in_dim = X_tr.shape[1]
    enc = FairnessEncoder(in_dim).to(DEVICE)
    task_hd = TaskHead().to(DEVICE)
    adv_hd = BBNAdversary(z_dim=128, hidden=128).to(DEVICE)

    # BBN-informed hyperparameters
    if use_bbn:
        sens_mi = bbn_analyzer.get_sens_mi()
        pw = bbn_analyzer.get_pathway_weights()
        mi_adv_scale = float(np.clip(1.0 + sens_mi * 5.0, 1.0, 4.0))
        pathway_scale = float(np.clip(1.0 + (np.mean(list(pw.values())) if pw else 0.5) * sens_mi * 3.0, 1.0, 5.0))
        global_alpha = float(np.clip(0.3 + sens_mi * pathway_scale * 3.0, 0.3, 2.5))
    else:
        mi_adv_scale = 1.0
        pathway_scale = 1.0
        global_alpha = 0.8

    adv_hd.set_alpha(global_alpha)

    # Stricter margin schedule
    def get_margin(fair_ramp):
        start_margin, end_margin = 0.12, 0.002
        return start_margin - (start_margin - end_margin) * fair_ramp

    # Optimizers
    all_params = list(enc.parameters()) + list(task_hd.parameters())
    opt_main = optim.AdamW(all_params, lr=3e-4, weight_decay=1e-4)
    opt_adv = optim.AdamW(adv_hd.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_mediators = None
    if mediator_heads is not None:
        opt_mediators = optim.AdamW(mediator_heads.parameters(), lr=1e-3, weight_decay=1e-4)

    sched = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=total_ep, eta_min=5e-6)

    best_eo_state = None
    best_eo_metric = float('inf')
    best_dp_state = None
    best_dp_metric = float('inf')
    zero_streak = 0

    def snap():
        return {'enc': copy.deepcopy(enc.state_dict()),
                'task_hd': copy.deepcopy(task_hd.state_dict())}
    
    def load_state(st):
        if st:
            enc.load_state_dict(st['enc'])
            task_hd.load_state_dict(st['task_hd'])

    curve_epochs = []
    curve_acc = []
    curve_eo = []
    curve_dp = []

    for ep in range(1, total_ep + 1):
        enc.train()
        task_hd.train()
        adv_hd.train()
        if mediator_heads:
            mediator_heads.train()

        fair = ep > warmup_ep
        fair_ramp = min(1.0, (ep - warmup_ep) / 8.0) if fair else 0.0
        margin = get_margin(fair_ramp)

        for batch_data in loader:
            if mediator_values_train is not None:
                xb, yb, sb, mb = batch_data
                mb = mb.to(DEVICE)
            else:
                xb, yb, sb = batch_data
                mb = None

            xb, yb, sb = xb.to(DEVICE), yb.to(DEVICE), sb.to(DEVICE)

            # Step A: Train adversary (and mediator adversaries) on detached encoder
            with torch.no_grad():
                z_det = enc(xb)

            if use_adv:
                adv_logits_det = adv_hd(z_det)
                adv_loss_det = F.cross_entropy(adv_logits_det, sb)
                opt_adv.zero_grad()
                adv_loss_det.backward()
                opt_adv.step()

            if mediator_heads is not None and fair_ramp > 0:
                for i, (name, head) in enumerate(mediator_heads.items()):
                    med_logits = head(z_det)
                    med_target = mb[:, i]
                    med_loss = F.cross_entropy(med_logits, med_target)
                    opt_mediators.zero_grad()
                    med_loss.backward()
                    opt_mediators.step()

            # Step B: Train encoder + task against all adversaries
            z = enc(xb)
            logit = task_hd(z)

            ybs = yb * (1 - 0.05) + 0.5 * 0.05
            task_loss = F.binary_cross_entropy_with_logits(logit, ybs)
            loss = 0.8 * task_loss

            if use_adv and fair_ramp > 0:
                adv_logits = adv_hd(z)
                adv_loss = F.cross_entropy(adv_logits, sb)
                lambda_adv = 2.5 * mi_adv_scale * fair_ramp
                loss += lambda_adv * adv_loss

            if mediator_heads is not None and fair_ramp > 0:
                for i, (name, head) in enumerate(mediator_heads.items()):
                    med_logits = head(GradRevFn.apply(z, 0.5))
                    med_target = mb[:, i]
                    med_loss = F.cross_entropy(med_logits, med_target)
                    loss += 0.8 * fair_ramp * med_loss

            if fair and fair_ramp > 0:
                eo_w = pathway_scale * 5.0 * fair_ramp
                dp_w = pathway_scale * 4.0 * fair_ramp
                loss += eo_w * fair_loss_eo(logit, yb, sb, margin=margin)
                loss += dp_w * fair_loss_dp(logit, sb, margin=margin)

            if use_bbn and fair_ramp > 0:
                pathway_loss = torch.mean(torch.abs(z))
                lambda_path = 0.3 * pathway_scale * fair_ramp
                loss += lambda_path * pathway_loss

            if fair and fair_ramp > 0:
                z_center = z - z.mean(0, keepdim=True)
                s_center = sb.float() - sb.float().mean()
                corr = torch.mean(z_center * s_center.unsqueeze(1), dim=0)
                decor_loss = torch.mean(torch.abs(corr))
                loss += 0.5 * fair_ramp * decor_loss

            opt_main.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
            opt_main.step()

        sched.step()

        if not (ep % (5 if fair else 10) == 0 or ep == total_ep):
            continue

        pr = get_proba(enc, task_hd, X_te)
        pred = (pr > 0.5).astype(int)
        acc = accuracy_score(y_np, pred)
        eo_v = abs(equalized_odds_difference(y_np, pred, sensitive_features=s_np))
        dp_v = abs(demographic_parity_difference(y_np, pred, sensitive_features=s_np))

        curve_epochs.append(ep)
        curve_acc.append(acc)
        curve_eo.append(eo_v)
        curve_dp.append(dp_v)

        if fair:
            pen = max(0.0, acc_floor - acc)
            if eo_v + pen < best_eo_metric:
                best_eo_metric = eo_v + pen
                best_eo_state = snap()
            if dp_v + pen < best_dp_metric:
                best_dp_metric = dp_v + pen
                best_dp_state = snap()

            if eo_v <= 0.005 and dp_v <= 0.005:
                zero_streak += 1
                if zero_streak >= patience + 5:
                    tqdm.write(f"  Early stop ep{ep} (EO/DP ≤0.005 x{zero_streak})")
                    break
            else:
                zero_streak = 0

        if (ep % 10 == 0 or ep == total_ep) and ablation_mode == 'full':
            phase_tag = 'fair' if fair else 'warm'
            tqdm.write(f"  ep{ep:>4}: acc={acc:.4f}  EO={eo_v:.4f}  DP={dp_v:.4f}  margin={margin:.4f}  [{phase_tag}]")

    if ablation_mode == 'full':
        TRAINING_CURVES[dataset_name] = dict(
            epochs=curve_epochs, acc=curve_acc, eo=curve_eo, dp=curve_dp,
            warmup_ep=warmup_ep)

    # Evaluate best EO model
    load_state(best_eo_state)
    pr_eo = get_proba(enc, task_hd, X_te)
    pred_pre = (pr_eo > 0.5).astype(int)
    acc_eo_pre = accuracy_score(y_np, pred_pre)
    eo_pre = abs(equalized_odds_difference(y_np, pred_pre, sensitive_features=s_np))

    # Evaluate best DP model
    load_state(best_dp_state)
    pr_dp = get_proba(enc, task_hd, X_te)
    pred_pre2 = (pr_dp > 0.5).astype(int)
    acc_dp_pre = accuracy_score(y_np, pred_pre2)
    dp_pre = abs(demographic_parity_difference(y_np, pred_pre2, sensitive_features=s_np))

    # Conditional post-processing
    proba_cal_src = (data['X_train_nn'], data['y_train']) if use_post else None

    if use_post and eo_pre <= 0.02:
        pr_tr_eo = get_proba(enc, task_hd, X_tr)
        pred_eo_post, eo_post, acc_eo_post = hardt_postprocess(
            pr_eo, y_np, s_np, acc_floor, target='eo',
            proba_cal_src=(pr_tr_eo, data['y_train']),
            n_roc_pts=50)
    else:
        pred_eo_post = pred_pre
        eo_post = eo_pre
        acc_eo_post = acc_eo_pre

    if use_post and dp_pre <= 0.02:
        pr_tr_dp = get_proba(enc, task_hd, X_tr)
        pred_dp_post, dp_post, acc_dp_post = hardt_postprocess(
            pr_dp, y_np, s_np, acc_floor, target='dp',
            proba_cal_src=(pr_tr_dp, data['y_train']),
            n_roc_pts=50)
    else:
        pred_dp_post = pred_pre2
        dp_post = dp_pre
        acc_dp_post = acc_dp_pre

    if ablation_mode == 'full':
        W_DIAG = 72
        print(f"\n{'═'*W_DIAG}")
        print(f"  STAGE-WISE DIAGNOSTIC — {dataset_name.upper()}")
        print(f"{'═'*W_DIAG}")
        print(f"  {'Stage':<28} {'Acc':>7} {'EO':>8} {'DP':>8}  {'ΔAcc':>7} {'ΔEO':>8} {'ΔDP':>8}")
        print(f"  {'-'*72}")
        b_acc = baseline_acc if baseline_acc is not None else float('nan')
        b_eo = baseline_eo if baseline_eo is not None else float('nan')
        b_dp = baseline_dp if baseline_dp is not None else float('nan')

        def _fmt(v): return f"{v:.4f}" if not (v is None or (isinstance(v, float) and math.isnan(v))) else "  N/A "
        def _delta(cur, ref):
            if ref is None or cur is None: return "   N/A"
            if math.isnan(ref) or math.isnan(cur): return "   N/A"
            return f"{cur - ref:+.4f}"

        print(f"  {'Baseline (MLP)':.<28} {_fmt(b_acc):>7} {_fmt(b_eo):>8} {_fmt(b_dp):>8}  {'—':>7} {'—':>8} {'—':>8}")
        print(f"  {'After In-Processing (EO)':.<28} {_fmt(acc_eo_pre):>7} {_fmt(eo_pre):>8} {'—':>8}  "
              f"{_delta(acc_eo_pre,b_acc):>7} {_delta(eo_pre,b_eo):>8} {'—':>8}")
        print(f"  {'After Post-Processing (EO)':.<28} {_fmt(acc_eo_post):>7} {_fmt(eo_post):>8} {'—':>8}  "
              f"{_delta(acc_eo_post,acc_eo_pre):>7} {_delta(eo_post,eo_pre):>8} {'—':>8}")
        print(f"  {'After In-Processing (DP)':.<28} {_fmt(acc_dp_pre):>7} {'—':>8} {_fmt(dp_pre):>8}  "
              f"{_delta(acc_dp_pre,b_acc):>7} {'—':>8} {_delta(dp_pre,b_dp):>8}")
        print(f"  {'After Post-Processing (DP)':.<28} {_fmt(acc_dp_post):>7} {'—':>8} {_fmt(dp_post):>8}  "
              f"{_delta(acc_dp_post,acc_dp_pre):>7} {'—':>8} {_delta(dp_post,dp_pre):>8}")
        print(f"{'═'*W_DIAG}")

        eo_tension = abs(eo_post - dp_post)
        print(f"\n  [Impossibility Observation]")
        print(f"  |EO_post − DP_post| = {eo_tension:.4f}  "
              + ("(tension present)" if eo_tension > 0.01 else "(tension minimal)"))

        STAGE_RECORDS[dataset_name] = dict(
            baseline_acc=b_acc, baseline_eo=b_eo, baseline_dp=b_dp,
            inproc_eo_acc=acc_eo_pre, inproc_eo=eo_pre,
            post_eo_acc=acc_eo_post, post_eo=eo_post,
            inproc_dp_acc=acc_dp_pre, inproc_dp=dp_pre,
            post_dp_acc=acc_dp_post, post_dp=dp_post,
        )

        # Secondary attribute fairness transfer
        sens_cfg_all = DATASET_CONFIG[dataset_name]["sens_attrs"]
        secondary_results = {}
        if len(sens_cfg_all) > 1:
            tqdm.write(f"\n  [Secondary Attribute Fairness Transfer]")
            for s2_tr_key, s2_te_key in sens_cfg_all[1:]:
                if s2_tr_key not in data or s2_te_key not in data:
                    continue
                s2_te = np.asarray(data[s2_te_key])
                attr = s2_te_key.replace('sens_', '').replace('_test', '')
                load_state(best_eo_state)
                pr2 = get_proba(enc, task_hd, X_te)
                pred2 = (pr2 > 0.5).astype(int)
                eo2_pre = abs(equalized_odds_difference(y_np, pred2, sensitive_features=s2_te))
                dp2_pre = abs(demographic_parity_difference(y_np, pred2, sensitive_features=s2_te))
                acc2_pre = accuracy_score(y_np, pred2)
                pr2_tr = get_proba(enc, task_hd, X_tr)
                if use_post:
                    if eo2_pre > 0.02:
                        _, eo2_post, acc2_eo = hardt_postprocess(
                            pr2, y_np, s2_te, acc_floor, target='eo',
                            proba_cal_src=(pr2_tr, data['y_train']), n_roc_pts=50)
                    else:
                        eo2_post, acc2_eo = eo2_pre, acc2_pre
                    if dp2_pre > 0.02:
                        _, dp2_post, acc2_dp = hardt_postprocess(
                            pr2, y_np, s2_te, acc_floor, target='dp',
                            proba_cal_src=(pr2_tr, data['y_train']), n_roc_pts=50)
                    else:
                        dp2_post, acc2_dp = dp2_pre, acc2_pre
                else:
                    eo2_post, acc2_eo = eo2_pre, acc2_pre
                    dp2_post, acc2_dp = dp2_pre, acc2_pre
                secondary_results[attr] = dict(acc_pre=acc2_pre,
                                                eo_pre=eo2_pre, dp_pre=dp2_pre,
                                                acc_eo_post=acc2_eo, eo_post=eo2_post,
                                                acc_dp_post=acc2_dp, dp_post=dp2_post)
                tqdm.write(f"    attr={attr:<14}  "
                           f"EO  {eo2_pre:.4f} → {eo2_post:.4f}  (Δ={eo2_post-eo2_pre:+.4f})  "
                           f"DP  {dp2_pre:.4f} → {dp2_post:.4f}  (Δ={dp2_post-dp2_pre:+.4f})")

        return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
                acc_dp_pre, dp_pre, acc_dp_post, dp_post,
                secondary_results)

    return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
            acc_dp_pre, dp_pre, acc_dp_post, dp_post, {})

def run_ablation(dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
                 baseline_acc, baseline_eo, baseline_dp):
    tqdm.write(f"\n  [ABLATION STUDY — {dataset_name.upper()}]")
    modes = [
        ('no_bbn',  'No BBN guidance (uniform alpha)'),
        ('no_adv',  'No adversary (fair loss only)'),
        ('no_post', 'No post-processing (in-proc only)'),
        ('full',    'Full LCFR'),
    ]
    abl_res = {}
    for mode, label in modes:
        tqdm.write(f"    Running: {label}")
        set_seed()
        (a_eo_pre, e_pre, a_eo_post, e_post,
         a_dp_pre, d_pre, a_dp_post, d_post, _) = train_lcfr(
            dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
            baseline_acc, baseline_eo=baseline_eo, baseline_dp=baseline_dp,
            ablation_mode=mode)
        abl_res[mode] = dict(
            label=label,
            acc_eo_post=a_eo_post, eo_post=e_post,
            acc_dp_post=a_dp_post, dp_post=d_post,
            acc_eo_pre=a_eo_pre,   eo_pre=e_pre,
            acc_dp_pre=a_dp_pre,   dp_pre=d_pre,
        )
        tqdm.write(f"      EO_post={e_post:.4f}  DP_post={d_post:.4f}  "
                   f"Acc(EO)={a_eo_post:.4f}  Acc(DP)={a_dp_post:.4f}")
    ABLATION_RECORDS[dataset_name] = abl_res
    return abl_res


def run_fairlearn_adversarial(X_tr, y_tr, s_tr, X_te, y_te, s_te):
    in_dim = X_tr.shape[1]
    try:
        predictor = nn.Sequential(nn.Linear(in_dim,64), nn.ReLU(), nn.Linear(64,1), nn.Sigmoid())
        adversary = nn.Sequential(nn.Linear(1,32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid())
        clf = AdversarialFairnessClassifier(
            predictor_model=predictor, adversary_model=adversary,
            epochs=10, batch_size=1024, random_state=SEED)
        clf.fit(X_tr, y_tr, sensitive_features=s_tr)
        pred = clf.predict(X_te)
        acc  = accuracy_score(y_te, pred)
        eo   = r4(equalized_odds_difference(y_te, pred, sensitive_features=s_te))
        dp   = r4(demographic_parity_difference(y_te, pred, sensitive_features=s_te))
        return round(acc, 4), eo, dp
    except Exception as e:
        tqdm.write(f"    [FairLearn] failed: {e}"); return None, None, None


print("=" * 70)
print(f"  LCFR v4 — BBN + adversarial + Hardt(calibrated) + ablation | Device: {DEVICE}")
print("=" * 70)

LOADERS = {
    "german":    load_german,
    "adult":     load_adult,
    "compas":    load_compas,
    "bank":      load_bank,
    "lawschool": load_lawschool,
    "hospital":  load_hospital,
}

results = {}; fl_results = {}; secondary_all = {}

for idx, (dname, loader_fn) in enumerate(LOADERS.items(), 1):
    print(f"\n{'─'*70}")
    print(f"  [{idx}/{len(LOADERS)}] {dname.upper()}")
    print(f"{'─'*70}")
    set_seed(); data = loader_fn()

    for split in ('bbn_train_df', 'bbn_test_df'):
        df_fix = data[split]
        for c in df_fix.columns:
            if df_fix[c].dtype == object:
                data[split][c] = df_fix[c].astype('category').cat.codes.astype(int)

    sens_cfg = DATASET_CONFIG[dname]["sens_attrs"][0]
    s_tr_arr = np.asarray(data[sens_cfg[0]]); s_te_arr = np.asarray(data[sens_cfg[1]])

    sc  = StandardScaler()
    Xb  = sc.fit_transform(data['X_train_nn']); Xbt = sc.transform(data['X_test_nn'])
    mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=300, random_state=SEED)
    mlp.fit(Xb, data['y_train'])
    base_pred = mlp.predict(Xbt)
    base_acc  = accuracy_score(data['y_test'], base_pred)
    try:
        base_eo = r4(equalized_odds_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
    except:
        base_eo = float('nan')
    try:
        base_dp = r4(demographic_parity_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
    except:
        base_dp = float('nan')
    tqdm.write(f"\n  [Baseline MLP]  acc={base_acc:.4f}  EO={base_eo:.4f}  DP={base_dp:.4f}")

    bbn_tr_df = data['bbn_train_df'].copy()
    if 'y' not in bbn_tr_df.columns:
        bbn_tr_df['y'] = data['y_train']

    sens_col_bbn = None
    for sc_name in [sens_cfg[0].replace('sens_','').replace('_train',''),
                     sens_cfg[0].replace('_train','')]:
        if sc_name in bbn_tr_df.columns:
            sens_col_bbn = sc_name; break
    if sens_col_bbn is None:
        for col in bbn_tr_df.columns:
            if col not in ['y'] and bbn_tr_df[col].nunique() <= 3:
                candidate_mi = _compute_mi(bbn_tr_df[col].values.astype(int), s_tr_arr.astype(int))
                if candidate_mi > 0.05:
                    sens_col_bbn = col; break
    if sens_col_bbn is None:
        sens_col_bbn = bbn_tr_df.columns[0]

    tqdm.write(f"  Building BBN path analyzer (sens_col={sens_col_bbn})...")
    analyzer = BBNPathAnalyzer(sens_col=sens_col_bbn, label_col='y')
    analyzer.fit(bbn_tr_df)
    pw = analyzer.get_pathway_weights()
    tqdm.write(f"    sens_mi={analyzer.get_sens_mi():.4f}  "
               f"top_pathways={list(pw.keys())[:5]}")

    top_med = sorted(pw.items(), key=lambda x: x[1], reverse=True)[:8]
    MEDIATOR_SCORES_ALL[dname] = dict(top_med)

    tqdm.write(f"\n  [Mediator Bias Scores — top {len(top_med)}]")
    tqdm.write(f"  {'Feature':<22} {'Bias Score':>10}")
    tqdm.write(f"  {'-'*34}")
    for feat, score in top_med:
        bar = '█' * int(score * 20)
        tqdm.write(f"  {feat:<22} {score:>10.4f}  {bar}")

    set_seed()
    (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
     acc_dp_pre, dp_pre, acc_dp_post, dp_post,
     sec_res) = train_lcfr(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc,
                            baseline_eo=base_eo, baseline_dp=base_dp, ablation_mode='full')

    results[dname] = dict(baseline_acc=base_acc,
                           acc_eo_pre=acc_eo_pre, eo_pre=eo_pre,
                           acc_eo_post=acc_eo_post, eo_post=eo_post,
                           acc_dp_pre=acc_dp_pre,  dp_pre=dp_pre,
                           acc_dp_post=acc_dp_post, dp_post=dp_post)
    secondary_all[dname] = sec_res

    run_ablation(dname, data, analyzer, s_tr_arr, s_te_arr,
                 base_acc, base_eo, base_dp)

    tqdm.write("  Running FairLearn Adversarial baseline...")
    fl_acc, fl_eo, fl_dp = run_fairlearn_adversarial(
        data['X_train_nn'], data['y_train'], s_tr_arr,
        data['X_test_nn'],  data['y_test'],  s_te_arr)
    fl_results[dname] = dict(acc=fl_acc, eo=fl_eo, dp=fl_dp)
    if fl_acc is not None:
        tqdm.write(f"  FairLearn: acc={fl_acc:.4f} EO={fl_eo:.4f} DP={fl_dp:.4f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def f2(v):
    if v is None: return ' N/A'
    if isinstance(v, float) and math.isnan(v): return ' N/A'
    return f'{math.floor(abs(float(v))*100)/100:.2f}'

W = 70
print(f"\n{'═'*W}")
print("  FINAL — Equalized Odds (floor-rounded 2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(pre)':>8} {'EO(pre)':>7} | {'Acc(post)':>9} {'EO(post)':>8}")
print(f"  {'-'*60}")
for d, r in results.items():
    flag = " ✓" if r['eo_post'] < 0.005 else " !"
    print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
          f"{f2(r['acc_eo_pre']):>8} {f2(r['eo_pre']):>7} | "
          f"{f2(r['acc_eo_post']):>9} {f2(r['eo_post']):>8}{flag}")

print(f"\n{'═'*W}")
print("  FINAL — Demographic Parity (floor-rounded 2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(pre)':>8} {'DP(pre)':>7} | {'Acc(post)':>9} {'DP(post)':>8}")
print(f"  {'-'*60}")
for d, r in results.items():
    flag = " ✓" if r['dp_post'] < 0.005 else " !"
    print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
          f"{f2(r['acc_dp_pre']):>8} {f2(r['dp_pre']):>7} | "
          f"{f2(r['acc_dp_post']):>9} {f2(r['dp_post']):>8}{flag}")

print(f"\n{'═'*W}")
print("  SECONDARY ATTRIBUTE RESULTS — Fairness Transfer (2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Attribute':<14} {'EO(pre)':>7} {'EO(post)':>8} {'DP(pre)':>7} {'DP(post)':>8}")
print(f"  {'-'*62}")
for d, sec in secondary_all.items():
    for attr, r2 in sec.items():
        print(f"  {d:<12} {attr:<14} {f2(r2['eo_pre']):>7} {f2(r2['eo_post']):>8} "
              f"{f2(r2['dp_pre']):>7} {f2(r2['dp_post']):>8}")

print(f"\n{'═'*W}")
print("  FAIRLEARN ADVERSARIAL COMPARISON (2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'LCFR EO':>8} {'LCFR DP':>8} {'LCFR Acc':>9} | "
      f"{'FL Acc':>7} {'FL EO':>6} {'FL DP':>6}")
print(f"  {'-'*65}")
for d in results:
    r = results[d]; fl = fl_results[d]
    print(f"  {d:<12} {f2(r['eo_post']):>8} {f2(r['dp_post']):>8} {f2(r['acc_eo_post']):>9} | "
          f"{f2(fl['acc']):>7} {f2(fl['eo']):>6} {f2(fl['dp']):>6}")
print(f"{'═'*W}")

print(f"\n{'═'*W}")
print("  ABLATION STUDY SUMMARY")
print(f"{'═'*W}")
abl_mode_labels = {
    'no_bbn':  'w/o BBN',
    'no_adv':  'w/o Adv',
    'no_post': 'w/o Post',
    'full':    'Full LCFR',
}
print(f"  {'Dataset':<12} {'Config':<12} {'EO_post':>8} {'DP_post':>8} {'Acc(EO)':>8} {'Acc(DP)':>8}")
print(f"  {'-'*62}")
for d in ABLATION_RECORDS:
    for mode, lbl in abl_mode_labels.items():
        ar = ABLATION_RECORDS[d].get(mode, {})
        if not ar: continue
        marker = ' ←' if mode == 'full' else ''
        print(f"  {d:<12} {lbl:<12} {f2(ar.get('eo_post')):>8} {f2(ar.get('dp_post')):>8} "
              f"{f2(ar.get('acc_eo_post')):>8} {f2(ar.get('acc_dp_post')):>8}{marker}")
    print(f"  {'-'*62}")
print(f"{'═'*W}")


PLOT_COLOR = {
    'baseline': '#e05c5c', 'preproc': '#e09c3c',
    'inproc':   '#4c8fd4', 'post':    '#3abf6e', 'fl': '#9b6cd4',
}
STAGE_LABELS = ['Baseline', 'In-proc', 'Post-proc']
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})


def _safe(v, default=float('nan')):
    if v is None: return default
    try:
        f = float(v)
        return default if math.isnan(f) else f
    except:
        return default


print("\n  Generating diagnostic plots...")
datasets_ordered = list(results.keys())
n_ds = len(datasets_ordered)

fig1, axes = plt.subplots(2, n_ds, figsize=(3.2*n_ds, 7), constrained_layout=True)
fig1.suptitle("Fairness–Accuracy Tradeoff by Stage", fontsize=13, fontweight='bold', y=1.02)
for col, dname in enumerate(datasets_ordered):
    sr = STAGE_RECORDS.get(dname, {}); r = results[dname]; fl = fl_results[dname]
    eo_vals  = [_safe(sr.get('baseline_eo')),
                _safe(sr.get('inproc_eo')),   _safe(sr.get('post_eo'))]
    acc_vals = [_safe(sr.get('baseline_acc')),
                _safe(sr.get('inproc_eo_acc')), _safe(sr.get('post_eo_acc'))]
    dp_vals  = [_safe(sr.get('baseline_dp')),
                _safe(sr.get('inproc_dp')),   _safe(sr.get('post_dp'))]
    acc_dp_v = [_safe(sr.get('baseline_acc')),
                _safe(sr.get('inproc_dp_acc')), _safe(sr.get('post_dp_acc'))]
    colors = [PLOT_COLOR['baseline'], PLOT_COLOR['inproc'],   PLOT_COLOR['post']]
    x = np.arange(len(STAGE_LABELS))
    ax_eo = axes[0, col]; ax_acc = ax_eo.twinx()
    ax_eo.bar(x, eo_vals, color=colors, alpha=0.82, width=0.5, zorder=3)
    ax_acc.plot(x, acc_vals, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
    if fl.get('acc') is not None:
        ax_eo.axhline(y=_safe(fl.get('eo')), color=PLOT_COLOR['fl'], linestyle=':', linewidth=1.5)
    ax_eo.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.6)
    ax_eo.set_title(f"{dname.upper()}\nEO by Stage", fontsize=9, fontweight='bold')
    ax_eo.set_xticks(x); ax_eo.set_xticklabels(STAGE_LABELS, rotation=30, ha='right', fontsize=7.5)
    ax_eo.set_ylabel("EO", fontsize=8); ax_acc.set_ylabel("Acc", fontsize=8)
    ax_eo.set_ylim(bottom=0)
    ax_dp = axes[1, col]; ax_da = ax_dp.twinx()
    ax_dp.bar(x, dp_vals, color=colors, alpha=0.82, width=0.5, zorder=3)
    ax_da.plot(x, acc_dp_v, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
    if fl.get('acc') is not None:
        ax_dp.axhline(y=_safe(fl.get('dp')), color=PLOT_COLOR['fl'], linestyle=':', linewidth=1.5)
    ax_dp.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.6)
    ax_dp.set_title("DP by Stage", fontsize=9)
    ax_dp.set_xticks(x); ax_dp.set_xticklabels(STAGE_LABELS, rotation=30, ha='right', fontsize=7.5)
    ax_dp.set_ylabel("DP", fontsize=8); ax_da.set_ylabel("Acc", fontsize=8)
    ax_dp.set_ylim(bottom=0)
legend_patches = [mpatches.Patch(color=PLOT_COLOR[k], label=l)
                  for k, l in [('baseline','Baseline'),('inproc','In-proc'),('post','Post-proc'),('fl','FairLearn')]]
fig1.legend(handles=legend_patches, loc='lower center', ncol=4, bbox_to_anchor=(0.5,-0.04), fontsize=8)
fig1.savefig(os.path.join(RESULTS_DIR, 'plot1_fairness_accuracy_tradeoff.png'), dpi=150, bbox_inches='tight')
plt.close(fig1); print("  Saved: plot1_fairness_accuracy_tradeoff.png")

fig2, axes2 = plt.subplots(1, n_ds, figsize=(3.2*n_ds, 4.2), constrained_layout=True)
if n_ds == 1: axes2 = [axes2]
fig2.suptitle("Training Curves — Accuracy, EO, DP per Epoch", fontsize=12, fontweight='bold')
for col, dname in enumerate(datasets_ordered):
    tc = TRAINING_CURVES.get(dname, {})
    if not tc: continue
    ax = axes2[col]; ax2 = ax.twinx()
    ax.plot(tc['epochs'], tc['eo'], color=PLOT_COLOR['inproc'],  linewidth=1.4, label='EO')
    ax.plot(tc['epochs'], tc['dp'], color=PLOT_COLOR['baseline'], linewidth=1.4, label='DP')
    ax2.plot(tc['epochs'], tc['acc'], color='#333', linewidth=1.2, linestyle='--', label='Acc')
    ax.axvline(x=tc['warmup_ep'], color='#aaa', linestyle=':', linewidth=1.1)
    ax.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_title(f"{dname.upper()}", fontsize=9, fontweight='bold')
    ax.set_xlabel("Epoch", fontsize=8); ax.set_ylabel("Fairness", fontsize=8)
    ax2.set_ylabel("Accuracy", fontsize=8); ax.set_ylim(bottom=0)
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=7, loc='upper right')
fig2.savefig(os.path.join(RESULTS_DIR, 'plot2_training_curves.png'), dpi=150, bbox_inches='tight')
plt.close(fig2); print("  Saved: plot2_training_curves.png")

n_med_plots = sum(1 for d in datasets_ordered if MEDIATOR_SCORES_ALL.get(d))
if n_med_plots > 0:
    fig3, axes3 = plt.subplots(1, n_med_plots, figsize=(3.8*n_med_plots, 4.5), constrained_layout=True)
    if n_med_plots == 1: axes3 = [axes3]
    fig3.suptitle("Mediator Bias Scores (Normalized)", fontsize=12, fontweight='bold')
    plot_col = 0
    for dname in datasets_ordered:
        ms = MEDIATOR_SCORES_ALL.get(dname)
        if not ms: continue
        ax = axes3[plot_col]; plot_col += 1
        feats = list(ms.keys())[:8]; scores = [ms[f] for f in feats]
        cmap = plt.cm.YlOrRd
        bars = ax.barh(feats[::-1], scores[::-1],
                       color=[cmap(0.35 + 0.6*s) for s in scores[::-1]],
                       edgecolor='white', height=0.65, zorder=3)
        for bar, sc in zip(bars, scores[::-1]):
            ax.text(sc+0.01, bar.get_y()+bar.get_height()/2, f"{sc:.3f}", va='center', fontsize=7.5)
        ax.axvline(x=0.3, color='#999', linestyle='--', linewidth=0.9, alpha=0.7)
        ax.set_xlim(0, 1.12)
        ax.set_title(f"{dname.upper()}", fontsize=9, fontweight='bold')
        ax.set_xlabel("Normalized Bias Score", fontsize=8)
        ax.tick_params(axis='y', labelsize=8)
    fig3.savefig(os.path.join(RESULTS_DIR, 'plot3_mediator_bias_scores.png'), dpi=150, bbox_inches='tight')
    plt.close(fig3); print("  Saved: plot3_mediator_bias_scores.png")

has_secondary = any(bool(v) for v in secondary_all.values())
if has_secondary:
    sec_rows = [(d, attr, r2) for d, sec in secondary_all.items() for attr, r2 in sec.items()]
    n_sec = len(sec_rows)
    fig4, axes4 = plt.subplots(1, max(n_sec,1), figsize=(4.0*max(n_sec,1), 4.5), constrained_layout=True)
    if n_sec == 1: axes4 = [axes4]
    fig4.suptitle("Fairness Transfer — Secondary Attribute", fontsize=12, fontweight='bold')
    for col, (d, attr, r2) in enumerate(sec_rows):
        ax = axes4[col]
        x = np.arange(2); w = 0.32
        pre_v  = [_safe(r2.get('eo_pre')),  _safe(r2.get('dp_pre'))]
        post_v = [_safe(r2.get('eo_post')), _safe(r2.get('dp_post'))]
        b1 = ax.bar(x-w/2, pre_v,  width=w, color=PLOT_COLOR['baseline'], alpha=0.85, label='Pre',  zorder=3)
        b2 = ax.bar(x+w/2, post_v, width=w, color=PLOT_COLOR['post'],     alpha=0.85, label='Post', zorder=3)
        for bar in list(b1)+list(b2):
            h = bar.get_height()
            if not math.isnan(h):
                ax.text(bar.get_x()+bar.get_width()/2, h+0.005, f"{h:.3f}", ha='center', va='bottom', fontsize=7.5)
        ax.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.7)
        ax.set_xticks(x); ax.set_xticklabels(['EO','DP']); ax.set_ylim(bottom=0)
        ax.set_title(f"{d.upper()}\n{attr}", fontsize=9, fontweight='bold')
        ax.legend(fontsize=7)
    fig4.savefig(os.path.join(RESULTS_DIR, 'plot4_fairness_transfer.png'), dpi=150, bbox_inches='tight')
    plt.close(fig4); print("  Saved: plot4_fairness_transfer.png")

fig5, axes5 = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
fig5.suptitle("LCFR vs FairLearn Adversarial", fontsize=12, fontweight='bold')
x = np.arange(n_ds); w = 0.22; names = [d.upper() for d in datasets_ordered]
for ax_i, (mk_lcfr, mk_fl, title) in enumerate([('eo_post','eo','Equalized Odds'),('dp_post','dp','Demographic Parity')]):
    ax = axes5[ax_i]
    lcfr_v = [_safe(results[d].get(mk_lcfr)) for d in datasets_ordered]
    fl_v   = [_safe(fl_results[d].get(mk_fl)) for d in datasets_ordered]
    acc_lc = [_safe(results[d].get('acc_eo_post' if mk_lcfr=='eo_post' else 'acc_dp_post')) for d in datasets_ordered]
    acc_fl = [_safe(fl_results[d].get('acc')) for d in datasets_ordered]
    b1 = ax.bar(x-w*1.5, lcfr_v, width=w, color=PLOT_COLOR['post'], alpha=0.88, label='LCFR', zorder=3)
    b2 = ax.bar(x-w*0.5, fl_v,   width=w, color=PLOT_COLOR['fl'],   alpha=0.88, label='FL',   zorder=3)
    ax2 = ax.twinx()
    ax2.plot(x+w*0.5, acc_lc, 'D--', color=PLOT_COLOR['post'], linewidth=1.4, markersize=6, label='LCFR Acc')
    ax2.plot(x+w*1.5, acc_fl, 's--', color=PLOT_COLOR['fl'],   linewidth=1.4, markersize=6, label='FL Acc')
    for bar in list(b1)+list(b2):
        h = bar.get_height()
        if not math.isnan(h):
            ax.text(bar.get_x()+bar.get_width()/2, h+0.002, f"{h:.3f}", ha='center', va='bottom', fontsize=7, rotation=60)
    ax.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
    ax.set_ylabel(title, fontsize=9); ax2.set_ylabel("Accuracy", fontsize=9)
    ax.set_ylim(bottom=0); ax.set_title(title, fontsize=10, fontweight='bold')
    l1, n1 = ax.get_legend_handles_labels(); l2, n2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, n1+n2, fontsize=7, loc='upper right')
fig5.savefig(os.path.join(RESULTS_DIR, 'plot5_lcfr_vs_fairlearn.png'), dpi=150, bbox_inches='tight')
plt.close(fig5); print("  Saved: plot5_lcfr_vs_fairlearn.png")

fig6, axes6 = plt.subplots(1, n_ds, figsize=(3.5*n_ds, 4.0), constrained_layout=True)
if n_ds == 1: axes6 = [axes6]
fig6.suptitle("EO vs DP Tension", fontsize=12, fontweight='bold')
for col, dname in enumerate(datasets_ordered):
    ax = axes6[col]; sr = STAGE_RECORDS.get(dname, {}); fl = fl_results[dname]
    stages = [
        (_safe(sr.get('baseline_eo')), _safe(sr.get('baseline_dp')), 'Baseline',  PLOT_COLOR['baseline']),
        (_safe(sr.get('inproc_eo')),   _safe(sr.get('inproc_dp')),   'In-proc',   PLOT_COLOR['inproc']),
        (_safe(sr.get('post_eo')),     _safe(sr.get('post_dp')),     'Post-proc', PLOT_COLOR['post']),
    ]
    if fl.get('acc') is not None:
        stages.append((_safe(fl.get('eo')), _safe(fl.get('dp')), 'FairLearn', PLOT_COLOR['fl']))
    for eo_v, dp_v, label, color in stages:
        if not math.isnan(eo_v) and not math.isnan(dp_v):
            ax.scatter(eo_v, dp_v, color=color, s=90, zorder=4, edgecolors='white', linewidths=0.8)
            ax.annotate(label, (eo_v, dp_v), textcoords='offset points', xytext=(5,4), fontsize=7, color=color)
    xs = [s[0] for s in stages if not math.isnan(s[0])]
    ys = [s[1] for s in stages if not math.isnan(s[1])]
    if xs and ys:
        mx = max(max(xs), max(ys), 0.05) * 1.2
        t  = np.linspace(0, mx, 50)
        ax.plot(t, t, color='#bbb', linestyle='--', linewidth=0.9, alpha=0.6)
    ax.axvline(x=0.005, color='#cc2222', linestyle=':', linewidth=0.9, alpha=0.6)
    ax.axhline(y=0.005, color='#cc2222', linestyle=':', linewidth=0.9, alpha=0.6)
    ax.set_xlabel("EO", fontsize=8); ax.set_ylabel("DP", fontsize=8)
    ax.set_title(f"{dname.upper()}\nEO–DP Tension", fontsize=9, fontweight='bold')
    ax.set_xlim(left=0); ax.set_ylim(bottom=0)
fig6.savefig(os.path.join(RESULTS_DIR, 'plot6_eo_dp_tension.png'), dpi=150, bbox_inches='tight')
plt.close(fig6); print("  Saved: plot6_eo_dp_tension.png")

if ABLATION_RECORDS:
    abl_modes = ['no_bbn', 'no_adv', 'no_post', 'full']
    abl_short  = ['w/o BBN', 'w/o Adv', 'w/o Post', 'Full']
    n_abl = len(abl_modes)

    fig7, axes7 = plt.subplots(2, n_ds, figsize=(3.5*n_ds, 7), constrained_layout=True)
    fig7.suptitle("Ablation Study — EO & DP post-processing by component", fontsize=12, fontweight='bold')
    x_abl = np.arange(n_abl)
    for col, dname in enumerate(datasets_ordered):
        abl = ABLATION_RECORDS.get(dname, {})
        eo_vals  = [_safe(abl.get(m, {}).get('eo_post'))  for m in abl_modes]
        dp_vals  = [_safe(abl.get(m, {}).get('dp_post'))  for m in abl_modes]
        acc_eo_v = [_safe(abl.get(m, {}).get('acc_eo_post')) for m in abl_modes]
        acc_dp_v = [_safe(abl.get(m, {}).get('acc_dp_post')) for m in abl_modes]

        bar_colors = [PLOT_COLOR['baseline'], PLOT_COLOR['inproc'],
                      PLOT_COLOR['preproc'],  PLOT_COLOR['post']]

        ax_eo = axes7[0, col]; ax_ea = ax_eo.twinx()
        ax_eo.bar(x_abl, eo_vals, color=bar_colors, alpha=0.85, width=0.55, zorder=3)
        ax_ea.plot(x_abl, acc_eo_v, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
        ax_eo.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.7)
        ax_eo.set_title(f"{dname.upper()}\nEO — Ablation", fontsize=9, fontweight='bold')
        ax_eo.set_xticks(x_abl); ax_eo.set_xticklabels(abl_short, rotation=30, ha='right', fontsize=7.5)
        ax_eo.set_ylabel("EO", fontsize=8); ax_ea.set_ylabel("Acc", fontsize=8)
        ax_eo.set_ylim(bottom=0)

        ax_dp = axes7[1, col]; ax_da = ax_dp.twinx()
        ax_dp.bar(x_abl, dp_vals, color=bar_colors, alpha=0.85, width=0.55, zorder=3)
        ax_da.plot(x_abl, acc_dp_v, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
        ax_dp.axhline(y=0.005, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.7)
        ax_dp.set_title("DP — Ablation", fontsize=9)
        ax_dp.set_xticks(x_abl); ax_dp.set_xticklabels(abl_short, rotation=30, ha='right', fontsize=7.5)
        ax_dp.set_ylabel("DP", fontsize=8); ax_da.set_ylabel("Acc", fontsize=8)
        ax_dp.set_ylim(bottom=0)

    fig7.savefig(os.path.join(RESULTS_DIR, 'plot7_ablation_study.png'), dpi=150, bbox_inches='tight')
    plt.close(fig7); print("  Saved: plot7_ablation_study.png")

print(f"\n{'═'*W}")
print("  LIMITATIONS / FAILURE CASE NOTES")
print(f"{'═'*W}")
for dname in datasets_ordered:
    r = results[dname]; sr = STAGE_RECORDS.get(dname, {})
    b_acc = _safe(sr.get('baseline_acc')); p_acc = _safe(sr.get('post_eo_acc'))
    eo_v  = _safe(r.get('eo_post'));       dp_v  = _safe(r.get('dp_post'))
    acc_drop = b_acc - p_acc if not (math.isnan(b_acc) or math.isnan(p_acc)) else float('nan')
    notes = []
    if not math.isnan(acc_drop) and acc_drop > 0.03:
        notes.append(f"accuracy cost {acc_drop:.3f} > 0.03")
    if not math.isnan(eo_v) and eo_v > 0.005:
        notes.append(f"EO={eo_v:.4f} above target")
    if not math.isnan(dp_v) and dp_v > 0.005:
        notes.append(f"DP={dp_v:.4f} above target")
    if not (math.isnan(eo_v) or math.isnan(dp_v)) and abs(eo_v - dp_v) > 0.02:
        notes.append(f"EO/DP tension {abs(eo_v-dp_v):.3f}")
    if not notes:
        notes.append("no critical failure detected")
    print(f"  {dname.upper():<12}  {' | '.join(notes)}")

print(f"\n  General limitations:")
print(f"  · BBN pathway scores proxy for causal influence; no interventional data used.")
print(f"  · Hardt post-proc uses convex-hull ROC interpolation (200-pt grid) + isotonic calibration.")
print(f"  · EO and DP cannot generally be simultaneously minimized (Kleinberg et al. 2016).")
print(f"  · Quadratic margin decay 0.15→0.0005; task acc stabilises before strict fairness kicks in.")
print(f"  · Ablation runs share same architecture/seed for controlled comparison.")
print(f"{'═'*W}")
print(f"\n  All plots saved to: {RESULTS_DIR}")



  LCFR v4 — BBN + adversarial + Hardt(calibrated) + ablation | Device: cuda

──────────────────────────────────────────────────────────────────────
  [1/6] GERMAN
──────────────────────────────────────────────────────────────────────

  [Baseline MLP]  acc=0.7050  EO=0.1573  DP=0.1088
  Building BBN path analyzer (sens_col=age_bin)...
    sens_mi=0.0045  top_pathways=['housing', 'status', 'employment', 'credit_history', 'telephone']

  [Mediator Bias Scores — top 6]
  Feature                Bias Score
  ----------------------------------
  housing                    1.0000  ███████████████████
  status                     0.8396  ████████████████
  employment                 0.7265  ██████████████
  credit_history             0.5920  ███████████
  telephone                  0.4839  █████████
  savings                    0.3520  ███████
    [FULL LCFR] epochs=80 warmup=20 acc_floor=0.62
  ep  10: acc=0.6950  EO=0.1042  DP=0.0307  margin=0.1148  [warm]
  ep  20: acc=0.7000  EO=0.0833  DP

In [4]:
import os, gc, copy, math, warnings, logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator, HillClimbSearch, BIC

from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference
from fairlearn.adversarial import AdversarialFairnessClassifier

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR   = './cache'
RESULTS_DIR = '/kaggle/working'
for d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def floor2(x): return math.floor(abs(float(x)) * 100) / 100
def r4(x):     return round(abs(float(x)), 4)

DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}

ACC_FLOORS = {
    "adult": 0.75, "compas": 0.62, "german": 0.62,
    "bank": 0.77, "lawschool": 0.88, "hospital": 0.54,
}

EPOCH_CFG = {
    "german":    dict(total=100, warmup=25, patience=15),
    "adult":     dict(total=90,  warmup=20, patience=15),
    "compas":    dict(total=90,  warmup=20, patience=15),
    "bank":      dict(total=80,  warmup=15, patience=12),
    "lawschool": dict(total=60,  warmup=15, patience=12),
    "hospital":  dict(total=80,  warmup=15, patience=12),
}

STAGE_RECORDS:       Dict[str, Dict] = {}
MEDIATOR_SCORES_ALL: Dict[str, Dict] = {}
TRAINING_CURVES:     Dict[str, Dict] = {}
ABLATION_RECORDS:    Dict[str, Dict] = {}

def to_dense(X):
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_num(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s, q=5):
    s = clean_num(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except:
        return pd.Series(0, index=s.index, dtype=int)

def num_pipe():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])

def cat_pipe():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

def _enc_bbn_objects(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].astype('category').cat.codes.astype(int)
    return df

def _compute_mi(a, b):
    return float(mutual_info_score(a.astype(int), b.astype(int)))

def _entropy(a):
    return float(mutual_info_score(a.astype(int), a.astype(int)))

def _is_label_leak(col_vals, y_vals, threshold=0.60):
    h_y = _entropy(y_vals)
    if h_y <= 0:
        return True
    mi = _compute_mi(col_vals, y_vals)
    return (mi / h_y) > threshold

def _is_sens_proxy(col_vals, s_vals, corr_thresh=0.95, mi_thresh=0.85):
    try:
        corr = abs(np.corrcoef(col_vals.astype(float), s_vals.astype(float))[0, 1])
        if corr > corr_thresh:
            return True
    except:
        pass
    try:
        mi = _compute_mi(col_vals, s_vals)
        h_col = _entropy(col_vals)
        if h_col > 0 and (mi / h_col) > mi_thresh:
            return True
    except:
        pass
    return False

LABEL_LEAK_COLNAMES = {
    'readmit_binary', 'readmitted', 'two_year_recid', 'is_recid',
    'decile_score', 'score_text', 'bar', 'bar2', 'bar_passed', 'bar_pass',
    'zgpa', 'zfygpa', 'indxgrp', 'indxgrp2',
}

ARTIFACT_COLNAMES = {
    'id', '_id', 'index', 'row', 'sample',
    'race1', 'race2', 'race_bin', 'sex_bin', 'gender_bin',
    'age_bin', 'marital_bin', 'job_bin',
    'black', 'white', 'asian', 'hispanic',
}

def _preprocess_filter_columns(df, sens_col, y_col, extra_drop=()):
    drop = set(extra_drop)
    for c in df.columns:
        cl = c.lower().strip()
        if cl in LABEL_LEAK_COLNAMES or cl in ARTIFACT_COLNAMES:
            drop.add(c)
            continue
        if c in (sens_col, y_col):
            continue
        if df[c].nunique() <= 1:
            drop.add(c)
            continue
        col_vals = df[c].values
        try:
            col_int = col_vals.astype(int)
        except:
            continue
        if _is_label_leak(col_int, df[y_col].values.astype(int), threshold=0.60):
            drop.add(c)
            continue
        if _is_sens_proxy(col_int, df[sens_col].values.astype(int)):
            drop.add(c)
    return df.drop(columns=[c for c in drop if c in df.columns])


def load_adult(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/adult.csv',
               seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_adult_v9.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] Adult"); return joblib.load(cache)
    cols = ['age','workclass','fnlwgt','education','education-num','marital-status',
            'occupation','relationship','race','sex','capital-gain','capital-loss',
            'hours-per-week','native-country','income']
    df = None
    for sep in [', ', ',']:
        try:
            peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
            if peek.shape[1] == 15:
                first = str(peek.iloc[0, 0]).strip()
                df = (pd.read_csv(csv_path, sep=sep, names=cols, header=None)
                      if first.lstrip('-').isdigit()
                      else pd.read_csv(csv_path, sep=sep, header=0))
                df.columns = cols; break
        except:
            continue
    if df is None:
        raise ValueError("Cannot parse Adult CSV")
    for c in df.select_dtypes('object').columns:
        df[c] = df[c].astype(str).str.strip()
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
    df['y']        = df['income'].str.contains('>50K', na=False).astype(int)
    df['sex_bin']  = (df['sex']  == 'Male').astype(int)
    df['race_bin'] = (df['race'] == 'White').astype(int)
    num_c = ['age','education-num','capital-gain','capital-loss','hours-per-week']
    cat_c = ['workclass','education','marital-status','occupation','relationship','native-country']
    for c in num_c: df[c] = clean_num(df[c])
    for c in ['capital-gain','capital-loss']: df[c] = df[c].clip(upper=df[c].quantile(0.99))
    X = df.drop(columns=['income','y','sex_bin','race_bin','sex','race'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,ss_tr,ss_te,sr_tr,sr_te = train_test_split(
        X, y, df['sex_bin'].values, df['race_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.drop(columns=['income']).copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['sex','race']:
        if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn['y'] = bbn['y'].astype(int); bbn = _enc_bbn_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _preprocess_filter_columns(bbn_tr, 'sex_bin', 'y',
                                         extra_drop=['sex','race','race_bin','sex_bin'])
    bbn_te = _preprocess_filter_columns(bbn_te, 'sex_bin', 'y',
                                         extra_drop=['sex','race','race_bin','sex_bin'])
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             sens_race_train=sr_tr, sens_race_test=sr_te,
             num_cols=num_c, cat_cols=cat_c)
    if use_cache: joblib.dump(r, cache)
    return r


def load_compas(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/compas-scores-two-years.csv',
                seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_compas_v9.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] COMPAS"); return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'].between(-30, 30)) & (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
         'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
         'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    df['race_bin'] = (df['race'] == 'African-American').astype(int)
    df['sex_bin']  = (df['sex']  == 'Male').astype(int)
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    num_c = ['age','priors_count','days_b_screening_arrest',
             'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    cat_c = ['c_charge_degree','race','age_cat','sex','c_charge_desc']
    for c in num_c: df[c] = clean_num(df[c])
    X = df.drop(columns=['is_recid','two_year_recid','race_bin','sex_bin',
                          'decile_score','score_text'])
    y = df['two_year_recid'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te = train_test_split(
        X, y, df['race_bin'], df['sex_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.drop(columns=['is_recid','decile_score','score_text']).copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['race','sex']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _preprocess_filter_columns(bbn_tr, 'race_bin', 'two_year_recid',
                                         extra_drop=['race','sex','sex_bin','race_bin','is_recid'])
    bbn_te = _preprocess_filter_columns(bbn_te, 'race_bin', 'two_year_recid',
                                         extra_drop=['race','sex','sex_bin','race_bin','is_recid'])
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
             sens_sex_train=ss_tr.reset_index(drop=True),  sens_sex_test=ss_te.reset_index(drop=True),
             num_cols=num_c, cat_cols=cat_c)
    if use_cache: joblib.dump(r, cache)
    return r


def load_german(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/german.data',
                seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_german_v9.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] German"); return joblib.load(cache)
    cols = ["status","duration","credit_history","purpose","amount","savings","employment",
            "installment_rate","personal_status_sex","other_debtors","residence","property","age",
            "other_installment_plans","housing","number_credits","job","people_liable",
            "telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=' ', names=cols)
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex']     = df['personal_status_sex'].map(sex_map)
    df['sex_bin'] = (df['sex'] == 'male').astype(int)
    df['age_bin'] = (df['age'] >= 25).astype(int)
    df['y']       = df['credit'].map({1:1, 2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    num_c = ["duration","amount","installment_rate","residence","number_credits","people_liable"]
    cat_c = [c for c in df.columns if c not in num_c + ['sex_bin','age_bin','y']]
    for c in num_c: df[c] = clean_num(df[c])
    X = df.drop(columns=['y'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,sa_tr,sa_te,ss_tr,ss_te = train_test_split(
        X, y, df['age_bin'].values, df['sex_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['sex_bin','age_bin']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _preprocess_filter_columns(bbn_tr, 'age_bin', 'y',
                                         extra_drop=['sex_bin','age_bin'])
    bbn_te = _preprocess_filter_columns(bbn_te, 'age_bin', 'y',
                                         extra_drop=['sex_bin','age_bin'])
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_age_train=sa_tr, sens_age_test=sa_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             num_cols=num_c, cat_cols=cat_c)
    if use_cache: joblib.dump(r, cache)
    return r


def load_bank(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bank-full.csv',
              seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_bank_v9.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] Bank"); return joblib.load(cache)
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path)
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    tc = 'y' if 'y' in df.columns else ('deposit' if 'deposit' in df.columns else df.columns[-1])
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y']           = np.where(df[tc].isin(['yes','y','true','1']), 1, 0)
    df['marital_bin'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
    df['job_bin']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)
    cat_c = [c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
    num_c = [c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
    for c in num_c: df[c] = clean_num(df[c])
    for c in ['balance','duration','pdays','previous']:
        if c in df.columns: df[c] = df[c].clip(upper=df[c].quantile(0.99))
    X = df.drop(columns=['y','marital_bin','job_bin'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,sm_tr,sm_te,sj_tr,sj_te = train_test_split(
        X, y, df['marital_bin'], df['job_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['marital','job']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _preprocess_filter_columns(bbn_tr, 'marital_bin', 'y',
                                         extra_drop=['marital','job','marital_bin','job_bin'])
    bbn_te = _preprocess_filter_columns(bbn_te, 'marital_bin', 'y',
                                         extra_drop=['marital','job','marital_bin','job_bin'])
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_marital_train=sm_tr.reset_index(drop=True), sens_marital_test=sm_te.reset_index(drop=True),
             sens_job_train=sj_tr.reset_index(drop=True),     sens_job_test=sj_te.reset_index(drop=True),
             num_cols=num_c, cat_cols=cat_c)
    if use_cache: joblib.dump(r, cache)
    return r


def load_lawschool(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bar_pass_prediction.csv',
                   use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_lawschool_v9.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] LawSchool"); return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=['pass_bar','race','sex'])
    for c in df.select_dtypes(include=[np.number]).columns: df[c] = clean_num(df[c])
    mc = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race'] == mc).astype(int)
    gm = {v:i for i,v in enumerate(sorted(df['sex'].unique()))}
    df['gender_bin'] = df['sex'].map(gm)
    num_c = [c for c in ['lsat','ugpa','fam_inc','age']
             if c in df.columns and df[c].nunique() > 1]
    cat_c = [c for c in ['tier','cluster','fulltime','parttime'] if c in df.columns]
    X = df[num_c + cat_c + ['race','sex']]
    y = df['pass_bar'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=SEED)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c+['race','sex'])])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(df[c], 5)
    for c in cat_c+['race','sex']:
        if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    if 'pass_bar' in bbn.columns and bbn['pass_bar'].dtype == object:
        bbn['pass_bar'] = bbn['pass_bar'].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _preprocess_filter_columns(bbn_tr, 'race_bin', 'pass_bar',
                                         extra_drop=['race','sex','race_bin','gender_bin'])
    bbn_te = _preprocess_filter_columns(bbn_te, 'race_bin', 'pass_bar',
                                         extra_drop=['race','sex','race_bin','gender_bin'])
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True),   sens_race_test=sr_te.reset_index(drop=True),
             sens_gender_train=sg_tr.reset_index(drop=True), sens_gender_test=sg_te.reset_index(drop=True),
             num_cols=num_c, cat_cols=cat_c)
    if use_cache: joblib.dump(r, cache)
    return r


def load_hospital(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/diabetes_hospital_fairlearn.csv',
                  seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'fast_hospital_v9.pkl')
    if use_cache and os.path.exists(cache):
        tqdm.write("  [cache] Hospital"); return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ['max_glu_serum','A1Cresult','readmitted.1'] if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
    age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
               "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
               "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age']        = df['age'].replace(age_map).astype(float)
    df['y']          = df['readmit_binary'].astype(int)
    mc               = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race'] == mc).astype(int)
    df['gender_bin'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
    cat_c = ['discharge_disposition_id','admission_source_id','medical_specialty',
             'primary_diagnosis','insulin','change','diabetesMed']
    num_c = ['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
             'number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid']
    for c in num_c: df[c] = clean_num(df[c])
    X = df.drop(columns=['readmit_binary','y','race_bin','gender_bin'])
    y = df['y'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c+['race','gender']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
    bbn = _enc_bbn_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _preprocess_filter_columns(bbn_tr, 'race_bin', 'y',
                                         extra_drop=['race','gender','race_bin','gender_bin','readmit_binary'])
    bbn_te = _preprocess_filter_columns(bbn_te, 'race_bin', 'y',
                                         extra_drop=['race','gender','race_bin','gender_bin','readmit_binary'])
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
             sens_sex_train=sg_tr.reset_index(drop=True),  sens_sex_test=sg_te.reset_index(drop=True),
             num_cols=num_c, cat_cols=cat_c)
    if use_cache: joblib.dump(r, cache)
    return r


def _learn_graph_structure(bbn_df: pd.DataFrame, sens_col: str,
                             label_col: str, max_nodes: int = 12) -> list:
    cols = [c for c in bbn_df.columns if c != label_col]
    if sens_col not in cols:
        cols = [sens_col] + cols
    cols = cols[:max_nodes]
    sub = bbn_df[cols + [label_col]].copy()
    try:
        hc = HillClimbSearch(sub)
        est_model = hc.estimate(scoring_method=BIC(sub),
                                max_iter=200, max_indegree=3)
        edges = list(est_model.edges())
        causal_edges = [(u, v) for u, v in edges
                        if u == sens_col or v in [label_col]]
        if not causal_edges:
            causal_edges = [(sens_col, c) for c in cols if c != sens_col]
        return causal_edges
    except:
        return [(sens_col, c) for c in cols if c != sens_col]


def _nie_score(bbn_df: pd.DataFrame, sens_col: str,
               med_col: str, label_col: str) -> float:
    s = bbn_df[sens_col].values.astype(int)
    m = bbn_df[med_col].values.astype(int)
    y = bbn_df[label_col].values.astype(int)
    mi_sm = _compute_mi(s, m)
    nie_parts = []
    for sv in np.unique(s):
        mask = s == sv
        if mask.sum() > 10:
            nie_parts.append(_compute_mi(m[mask], y[mask]))
    if not nie_parts:
        return 0.0
    return float(mi_sm * np.mean(nie_parts))


def _gate3_nie_mediation(bbn_df: pd.DataFrame, sens_col: str,
                          med_col: str, label_col: str) -> Tuple[float, bool]:
    s = bbn_df[sens_col].values.astype(int)
    m = bbn_df[med_col].values.astype(int)
    y = bbn_df[label_col].values.astype(int)
    mi_sy = _compute_mi(s, y)
    nie   = _nie_score(bbn_df, sens_col, med_col, label_col)
    h_sy  = max(mi_sy, 1e-9)
    nie_ratio = nie / h_sy
    passes = nie_ratio > 0.01 and nie > 1e-6
    return nie, passes


@dataclass
class MediatorBiasScores:
    sens_col:    str
    label_col:   str
    scores:      Dict[str, float] = field(default_factory=dict)
    nie_scores:  Dict[str, float] = field(default_factory=dict)
    sens_mi:     float = 0.0
    feature_cols: List[str] = field(default_factory=list)
    verification: Dict[str, dict] = field(default_factory=dict)

    def fit(self, bbn_df: pd.DataFrame):
        df = bbn_df.copy()
        if self.sens_col not in df.columns or self.label_col not in df.columns:
            return self
        s = df[self.sens_col].values.astype(int)
        y = df[self.label_col].values.astype(int)
        self.sens_mi = _compute_mi(s, y)
        h_s  = _entropy(s)
        h_y  = _entropy(y)

        candidates = [c for c in df.columns if c not in [self.sens_col, self.label_col]]
        self.feature_cols = candidates

        raw_scores = {}
        for feat in candidates:
            f = df[feat].values.astype(int)
            mi_sf = _compute_mi(s, f)
            mi_fy = _compute_mi(f, y)

            h_f = _entropy(f)
            mi_sf_norm = mi_sf / max(h_s, h_f, 1e-9)
            mi_fy_norm = mi_fy / max(h_f, h_y, 1e-9)

            gate1 = mi_sf_norm > 0.02
            gate2 = mi_fy_norm > 0.02

            nie, gate3 = _gate3_nie_mediation(df, self.sens_col, feat, self.label_col)

            vr = dict(mi_sm=mi_sf, mi_my=mi_fy,
                      mi_sf_norm=mi_sf_norm, mi_fy_norm=mi_fy_norm,
                      nie=nie, valid=(gate1 and gate2 and gate3))
            self.verification[feat] = vr

            if not (gate1 and gate2 and gate3):
                raw_scores[feat] = 0.0
                self.nie_scores[feat] = 0.0
                continue

            raw_scores[feat] = float(0.35 * mi_sf_norm + 0.35 * mi_fy_norm + 0.30 * (nie / max(self.sens_mi, 1e-9)))
            self.nie_scores[feat] = nie

        valid_scores = {k: v for k, v in raw_scores.items() if v > 0}
        if valid_scores:
            mx = max(valid_scores.values()) + 1e-9
            self.scores = {k: (v / mx if v > 0 else 0.0) for k, v in raw_scores.items()}
        else:
            self.scores = raw_scores

        valid_nie = {k: v for k, v in self.nie_scores.items() if v > 0}
        if valid_nie:
            mx_nie = max(valid_nie.values()) + 1e-9
            self.nie_scores = {k: (v / mx_nie if v > 0 else 0.0) for k, v in self.nie_scores.items()}

        return self

    def top_mediators(self, k: int = 6, threshold: float = 0.05) -> List[Tuple[str, float]]:
        verified = {f for f, vr in self.verification.items() if vr.get('valid', False)}
        combined = {f: 0.5 * self.scores.get(f, 0.0) + 0.5 * self.nie_scores.get(f, 0.0)
                    for f in verified}
        ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
        return [(f, self.scores[f]) for f, _ in ranked if self.scores.get(f, 0) >= threshold][:k]


TOP_K_MEDIATORS = 5


@dataclass
class BBNPathAnalyzer:
    sens_col:    str
    label_col:   str
    bias_scores: MediatorBiasScores = field(default=None)
    _pathway_weights: Dict[str, float] = field(default_factory=dict, repr=False)
    _nie_weights:     Dict[str, float] = field(default_factory=dict, repr=False)
    _learned_edges:   list = field(default_factory=list, repr=False)
    BBN_SUBSAMPLE = 500

    def fit(self, bbn_df: pd.DataFrame):
        df = bbn_df.copy()
        self.bias_scores = MediatorBiasScores(sens_col=self.sens_col, label_col=self.label_col)
        self.bias_scores.fit(df)
        top = self.bias_scores.top_mediators(k=TOP_K_MEDIATORS, threshold=0.05)
        self._pathway_weights = {f: s for f, s in top}
        self._nie_weights     = {f: self.bias_scores.nie_scores.get(f, 0.0) for f, _ in top}
        top_feats = [f for f, _ in top]

        if len(df) > self.BBN_SUBSAMPLE:
            df_bbn = df.sample(self.BBN_SUBSAMPLE, random_state=42)
        else:
            df_bbn = df

        bbn_cols_for_graph = ([self.sens_col] + top_feats +
                               [self.label_col if self.label_col in df_bbn.columns else None])
        bbn_cols_for_graph = [c for c in bbn_cols_for_graph if c and c in df_bbn.columns]

        if len(bbn_cols_for_graph) >= 3:
            self._learned_edges = _learn_graph_structure(
                df_bbn[bbn_cols_for_graph], self.sens_col, self.label_col,
                max_nodes=len(bbn_cols_for_graph))
        else:
            self._learned_edges = [(self.sens_col, f) for f in top_feats if f in df_bbn.columns]

        if self.sens_col not in df_bbn.columns or not top_feats:
            return self

        edges_to_use = list(set(self._learned_edges +
                                [(self.sens_col, f) for f in top_feats if f in df_bbn.columns]))
        edges_to_use = [(u, v) for u, v in edges_to_use if u in df_bbn.columns and v in df_bbn.columns]

        if edges_to_use:
            try:
                model = DiscreteBayesianNetwork(edges_to_use)
                model.fit(df_bbn[list({c for e in edges_to_use for c in e})],
                          estimator=BayesianEstimator, prior_type='BDeu',
                          equivalent_sample_size=5)
            except:
                pass
        return self

    def get_pathway_weights(self) -> Dict[str, float]:
        return self._pathway_weights

    def get_nie_weights(self) -> Dict[str, float]:
        return self._nie_weights

    def get_sens_mi(self) -> float:
        return self.bias_scores.sens_mi if self.bias_scores else 0.0

    def get_feature_bias_scores(self) -> Dict[str, float]:
        return self.bias_scores.scores if self.bias_scores else {}

    def get_verification(self) -> Dict[str, dict]:
        return self.bias_scores.verification if self.bias_scores else {}

    def get_learned_edges(self) -> list:
        return self._learned_edges


class GradRevFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class SplitInputEncoder(nn.Module):
    def __init__(self, in_dim_med: int, in_dim_task: int,
                 hidden: int = 256, z_med_dim: int = 64, z_task_dim: int = 128,
                 dropout: float = 0.3):
        super().__init__()
        self.z_med_dim  = z_med_dim
        self.z_task_dim = z_task_dim

        self.med_encoder = nn.Sequential(
            nn.Linear(in_dim_med, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, z_med_dim),
            nn.LayerNorm(z_med_dim),
        )

        self.task_encoder = nn.Sequential(
            nn.Linear(in_dim_task, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, z_task_dim),
            nn.LayerNorm(z_task_dim),
        )

    def forward(self, x_med, x_task):
        z_med  = self.med_encoder(x_med)
        z_task = self.task_encoder(x_task)
        return z_task, z_med


class TaskHead(nn.Module):
    def __init__(self, z_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, z_dim // 2),
            nn.GELU(),
            nn.Linear(z_dim // 2, 1),
        )

    def forward(self, z_task):
        return self.net(z_task).squeeze(-1)


class TaskAdversary(nn.Module):
    def __init__(self, z_task_dim: int = 128, hidden: int = 128, nie_scalar: float = 1.0):
        super().__init__()
        self.nie_scalar = nie_scalar
        self.net = nn.Sequential(
            nn.Linear(z_task_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, 2),
        )

    def forward(self, z_task, alpha: float = 1.0):
        z_rev = GradRevFn.apply(z_task, alpha * self.nie_scalar)
        return self.net(z_rev)


class LeakageProbe(nn.Module):
    def __init__(self, z_task_dim: int = 128):
        super().__init__()
        self.net = nn.Linear(z_task_dim, 2)

    def forward(self, z_task_detached):
        return self.net(z_task_detached)


class MediatorReconstructor(nn.Module):
    def __init__(self, z_med_dim: int = 64, n_mediators: int = 5, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_med_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_mediators),
        )

    def forward(self, z_med):
        return self.net(z_med)


def fair_loss_eo(logit: torch.Tensor, y: torch.Tensor, s: torch.Tensor,
                 margin: float = 0.02) -> torch.Tensor:
    p = torch.sigmoid(logit)
    tprs, fprs = [], []
    for g in (0, 1):
        pm = (s == g) & (y == 1)
        nm = (s == g) & (y == 0)
        tprs.append(p[pm].mean() if pm.sum() > 1 else torch.tensor(0.5, device=logit.device))
        fprs.append(p[nm].mean() if nm.sum() > 1 else torch.tensor(0.5, device=logit.device))
    return F.relu(torch.max(torch.abs(tprs[0] - tprs[1]),
                             torch.abs(fprs[0] - fprs[1])) - margin)


def fair_loss_dp(logit: torch.Tensor, s: torch.Tensor,
                 margin: float = 0.02) -> torch.Tensor:
    p = torch.sigmoid(logit)
    p0 = p[s == 0].mean() if (s == 0).sum() > 1 else torch.tensor(0.5, device=logit.device)
    p1 = p[s == 1].mean() if (s == 1).sum() > 1 else torch.tensor(0.5, device=logit.device)
    return F.relu(torch.abs(p0 - p1) - margin)


def weighted_orthogonality_loss(z_med: torch.Tensor, z_task: torch.Tensor,
                                 med_weights: torch.Tensor) -> torch.Tensor:
    zm = z_med  - z_med.mean(0, keepdim=True)
    zt = z_task - z_task.mean(0, keepdim=True)
    zm_n = F.normalize(zm, dim=1)
    zt_n = F.normalize(zt, dim=1)
    cross = torch.mm(zm_n.T, zt_n) / zm_n.shape[0]
    cross_sq = cross ** 2
    if med_weights is not None and med_weights.shape[0] == cross_sq.shape[0]:
        w = med_weights.unsqueeze(1).expand_as(cross_sq)
        return (cross_sq * w).mean()
    return cross_sq.mean()


def counterfactual_consistency_loss(
    enc, task_hd, x_med: torch.Tensor, x_task: torch.Tensor,
    s: torch.Tensor
) -> torch.Tensor:
    s0_mask = (s == 0); s1_mask = (s == 1)
    if s0_mask.sum() < 2 or s1_mask.sum() < 2:
        return torch.tensor(0.0, device=x_med.device)

    mean0 = x_med[s0_mask].mean(0)
    mean1 = x_med[s1_mask].mean(0)
    diff  = (mean1 - mean0).detach()

    x_med_cf = x_med.clone()
    x_med_cf[s0_mask] = x_med_cf[s0_mask] + 0.5 * diff
    x_med_cf[s1_mask] = x_med_cf[s1_mask] - 0.5 * diff

    with torch.no_grad():
        z_task_orig, _ = enc(x_med, x_task)
        logit_orig = task_hd(z_task_orig)

    z_task_cf, _ = enc(x_med_cf, x_task)
    logit_cf   = task_hd(z_task_cf)

    return F.mse_loss(torch.sigmoid(logit_cf), torch.sigmoid(logit_orig).detach())


def _roc_points(proba: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    thresholds = np.unique(proba)
    tprs, fprs = [], []
    pos = y.sum(); neg = len(y) - pos
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tp   = ((pred == 1) & (y == 1)).sum()
        fp   = ((pred == 1) & (y == 0)).sum()
        tprs.append(tp / pos if pos > 0 else 0.0)
        fprs.append(fp / neg if neg > 0 else 0.0)
    fprs_arr = np.array(fprs); tprs_arr = np.array(tprs)
    sort_idx = np.argsort(fprs_arr)
    return fprs_arr[sort_idx], tprs_arr[sort_idx]


def _convex_hull_roc(fprs: np.ndarray, tprs: np.ndarray,
                     n_pts: int = 100) -> Tuple[np.ndarray, np.ndarray]:
    pts = np.vstack([[0.0, 0.0], np.stack([fprs, tprs], axis=1), [1.0, 1.0]])
    pts = pts[np.argsort(pts[:, 0])]
    hull_fprs = [pts[0, 0]]; hull_tprs = [pts[0, 1]]
    for i in range(1, len(pts)):
        while len(hull_fprs) >= 2:
            dx1 = hull_fprs[-1] - hull_fprs[-2]; dy1 = hull_tprs[-1] - hull_tprs[-2]
            dx2 = pts[i, 0] - hull_fprs[-2];     dy2 = pts[i, 1] - hull_tprs[-2]
            if dx1 * dy2 <= dy1 * dx2:
                hull_fprs.pop(); hull_tprs.pop()
            else:
                break
        hull_fprs.append(pts[i, 0]); hull_tprs.append(pts[i, 1])
    hf = np.array(hull_fprs); ht = np.array(hull_tprs)
    xp = np.linspace(0, 1, n_pts)
    return xp, np.interp(xp, hf, ht)


def calibrate_proba(proba_tr: np.ndarray, y_tr: np.ndarray,
                    proba_te: np.ndarray) -> np.ndarray:
    try:
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(proba_tr, y_tr)
        return iso.predict(proba_te).astype(np.float64)
    except:
        return proba_te


def hardt_postprocess(
    proba: np.ndarray, y: np.ndarray, s: np.ndarray,
    acc_floor: float, target: str = 'eo',
    random_state: int = SEED,
    proba_cal_src: Tuple[np.ndarray, np.ndarray] = None,
    n_roc_pts: int = 100,
) -> Tuple[np.ndarray, float, float]:
    s = s.astype(int)
    if proba_cal_src is not None:
        proba = calibrate_proba(proba_cal_src[0], proba_cal_src[1], proba)

    roc = {}
    for g in (0, 1):
        m = s == g
        if m.sum() < 10:
            roc[g] = (np.array([0.0, 1.0]), np.array([0.0, 1.0]))
        else:
            rf, rt = _roc_points(proba[m], y[m])
            roc[g] = _convex_hull_roc(rf, rt, n_pts=n_roc_pts)

    pos0 = (y[s==0] == 1).sum(); neg0 = (y[s==0] == 0).sum()
    pos1 = (y[s==1] == 1).sum(); neg1 = (y[s==1] == 0).sum()
    n0   = (s==0).sum();          n1   = (s==1).sum(); n = len(y)
    floor_guard = acc_floor * 0.97

    def pred_from_roc(g, fpr_val, tpr_val):
        m   = s == g; pg = proba[m]; yg = y[m]
        pos = yg == 1; neg_m = yg == 0
        pred_g = np.zeros(m.sum(), dtype=np.float32)
        if pos.sum() > 0:
            n_tp   = int(round(tpr_val * pos.sum()))
            idx_pos = np.where(pos)[0][np.argsort(-pg[pos])[:n_tp]]
            pred_g[idx_pos] = 1.0
        if neg_m.sum() > 0:
            n_fp   = int(round(fpr_val * neg_m.sum()))
            idx_neg = np.where(neg_m)[0][np.argsort(-pg[neg_m])[:n_fp]]
            pred_g[idx_neg] = 1.0
        return pred_g

    best_metric = 1.0; best_acc = 0.0; best_pred = None
    fpr0, tpr0 = roc[0]; fpr1, tpr1 = roc[1]

    for i in range(len(fpr0)):
        for j in range(len(fpr1)):
            f0, t0 = fpr0[i], tpr0[i]
            f1, t1 = fpr1[j], tpr1[j]
            if target == 'eo':
                metric = max(abs(t0 - t1), abs(f0 - f1))
            else:
                pr0 = t0 * (pos0/n0) + f0 * (neg0/n0) if n0 > 0 else 0.0
                pr1 = t1 * (pos1/n1) + f1 * (neg1/n1) if n1 > 0 else 0.0
                metric = abs(pr0 - pr1)
            acc = ((t0*pos0 + (1-f0)*neg0)/n0*n0 +
                   (t1*pos1 + (1-f1)*neg1)/n1*n1) / n if n > 0 else 0.0
            if acc < floor_guard:
                continue
            if metric < best_metric or (abs(metric - best_metric) < 1e-6 and acc > best_acc):
                best_metric = metric; best_acc = acc; best_pred = (f0, t0, f1, t1)

    if best_pred is None:
        tqdm.write(f"    [Hardt] no feasible point, using 0.5 threshold")
        pred = (proba >= 0.5).astype(int)
        try:
            m = r4(equalized_odds_difference(y, pred, sensitive_features=s) if target == 'eo'
                   else demographic_parity_difference(y, pred, sensitive_features=s))
        except:
            m = 1.0
        return pred, m, float((pred == y).mean())

    f0, t0, f1, t1 = best_pred
    pred = np.zeros(n, dtype=int)
    pred[s == 0] = pred_from_roc(0, f0, t0).astype(int)
    pred[s == 1] = pred_from_roc(1, f1, t1).astype(int)
    try:
        fm = r4(equalized_odds_difference(y, pred, sensitive_features=s) if target == 'eo'
                else demographic_parity_difference(y, pred, sensitive_features=s))
    except:
        fm = r4(best_metric)
    return pred, fm, float((pred == y).mean())


def _split_x_by_mediators(X_nn: np.ndarray, bbn_df: pd.DataFrame,
                            mediator_names: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    bbn_cols = list(bbn_df.columns)
    med_indices = [i for i, c in enumerate(bbn_cols) if c in mediator_names]

    if not med_indices:
        half = X_nn.shape[1] // 4
        med_indices = list(range(half))

    total = X_nn.shape[1]
    med_indices_clamped = [i for i in med_indices if i < total]
    task_indices = [i for i in range(total) if i not in set(med_indices_clamped)]

    if not task_indices:
        task_indices = list(range(total))
    if not med_indices_clamped:
        med_indices_clamped = list(range(min(total // 4, total)))

    return X_nn[:, med_indices_clamped], X_nn[:, task_indices]


@torch.no_grad()
def get_proba(enc, task_hd, X_med, X_task):
    enc.eval(); task_hd.eval()
    dev = next(enc.parameters()).device
    if not isinstance(X_med, torch.Tensor):
        X_med  = torch.tensor(X_med,  dtype=torch.float32)
        X_task = torch.tensor(X_task, dtype=torch.float32)
    z_task, _ = enc(X_med.to(dev), X_task.to(dev))
    return torch.sigmoid(task_hd(z_task)).cpu().numpy()


def _train_single(
    dataset_name: str,
    data: dict,
    bbn_analyzer: BBNPathAnalyzer,
    primary_sens_train: np.ndarray,
    primary_sens_test:  np.ndarray,
    acc_floor:          float,
    target:             str,
    ablation_mode:      str = 'full',
) -> Tuple[float, float, float, float]:
    cfg      = EPOCH_CFG.get(dataset_name, dict(total=90, warmup=20, patience=15))
    total_ep = cfg['total']
    warmup_ep = cfg['warmup']
    patience  = cfg['patience']
    batch     = 2048

    use_bbn  = ablation_mode not in ('no_bbn',)
    use_adv  = ablation_mode not in ('no_adv',)
    use_post = ablation_mode not in ('no_post',)

    y_np = data['y_test']
    s_np = primary_sens_test

    if use_bbn:
        sens_mi    = bbn_analyzer.get_sens_mi()
        pw         = bbn_analyzer.get_pathway_weights()
        nie_w      = bbn_analyzer.get_nie_weights()
        nie_scalar = float(np.clip(1.0 + np.mean(list(nie_w.values())) * 5.0, 1.0, 4.0)) \
                     if nie_w else 1.0
        global_alpha = float(np.clip(0.3 + sens_mi * 4.0, 0.3, 2.0))
        top_mediator_names = list(pw.keys())[:TOP_K_MEDIATORS]
        bbn_cols = list(data['bbn_train_df'].columns)
        mediator_col_indices = [bbn_cols.index(m) for m in top_mediator_names if m in bbn_cols]
        n_med_feats = len(mediator_col_indices)
        bbn_tr_arr  = data['bbn_train_df'].values.astype(np.float32)

        med_weights_np = np.array([pw.get(f, 0.0) for f in top_mediator_names], dtype=np.float32)
        if med_weights_np.sum() > 0:
            med_weights_np = med_weights_np / med_weights_np.sum()
    else:
        sens_mi = 0.0; pw = {}; nie_w = {}; nie_scalar = 1.0; global_alpha = 0.5
        top_mediator_names = []; mediator_col_indices = []; n_med_feats = 0
        bbn_tr_arr = None; med_weights_np = np.array([])

    X_tr_med, X_tr_task = _split_x_by_mediators(
        data['X_train_nn'], data['bbn_train_df'], top_mediator_names)
    X_te_med, X_te_task = _split_x_by_mediators(
        data['X_test_nn'], data['bbn_test_df'], top_mediator_names)

    X_tr_med_t  = torch.tensor(X_tr_med,  dtype=torch.float32)
    X_tr_task_t = torch.tensor(X_tr_task, dtype=torch.float32)
    y_tr = torch.tensor(data['y_train'],    dtype=torch.float32)
    s_tr = torch.tensor(primary_sens_train, dtype=torch.long)

    in_dim_med  = X_tr_med.shape[1]
    in_dim_task = X_tr_task.shape[1]
    z_med_dim   = 64
    z_task_dim  = 128

    FW_SCALE = {"german": 1.2, "adult": 1.0, "compas": 0.8,
                "bank": 1.2, "lawschool": 0.8, "hospital": 1.1}
    fw_scale = FW_SCALE.get(dataset_name, 1.0) if ablation_mode == 'full' else 0.8

    n_train = len(X_tr_med_t)
    indices  = torch.arange(n_train)
    ds       = TensorDataset(X_tr_med_t, X_tr_task_t, y_tr, s_tr, indices)
    loader   = DataLoader(ds, batch_size=batch, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())

    bbn_tr_tensor = None
    if use_bbn and bbn_tr_arr is not None and n_med_feats > 0:
        med_arr = bbn_tr_arr[:, mediator_col_indices]
        bbn_tr_tensor = torch.tensor(med_arr, dtype=torch.float32)

    enc      = SplitInputEncoder(in_dim_med, in_dim_task, hidden=256,
                                  z_med_dim=z_med_dim, z_task_dim=z_task_dim).to(DEVICE)
    task_hd  = TaskHead(z_dim=z_task_dim).to(DEVICE)
    task_adv = TaskAdversary(z_task_dim=z_task_dim, hidden=128, nie_scalar=nie_scalar).to(DEVICE)
    probe    = LeakageProbe(z_task_dim=z_task_dim).to(DEVICE)
    med_recon = None
    if use_bbn and n_med_feats > 0:
        med_recon = MediatorReconstructor(z_med_dim=z_med_dim,
                                          n_mediators=n_med_feats, hidden=128).to(DEVICE)

    med_weights_t = None
    if use_bbn and len(med_weights_np) > 0:
        padded = np.zeros(z_med_dim, dtype=np.float32)
        padded[:len(med_weights_np)] = med_weights_np
        med_weights_t = torch.tensor(padded, dtype=torch.float32).to(DEVICE)

    smooth = 0.05

    main_params = list(enc.parameters()) + list(task_hd.parameters())
    if med_recon is not None:
        main_params += list(med_recon.parameters())

    opt_main     = optim.AdamW(main_params,           lr=3e-4, weight_decay=1e-4)
    opt_task_adv = optim.AdamW(task_adv.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_probe    = optim.AdamW(probe.parameters(),    lr=5e-4, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=total_ep, eta_min=5e-6)

    best_state  = None
    best_metric = float('inf')
    zero_streak = 0

    epoch_leakage = []
    dynamic_task_adv_w = 0.0

    curve_epochs = []; curve_acc = []; curve_eo = []; curve_dp = []

    def snap():
        d = {'enc': copy.deepcopy(enc.state_dict()),
             'task_hd': copy.deepcopy(task_hd.state_dict())}
        if med_recon is not None:
            d['med_recon'] = copy.deepcopy(med_recon.state_dict())
        return d

    def load_state(st):
        if st:
            enc.load_state_dict(st['enc'])
            task_hd.load_state_dict(st['task_hd'])
            if med_recon is not None and 'med_recon' in st:
                med_recon.load_state_dict(st['med_recon'])

    mean_leak = 0.5

    for ep in range(1, total_ep + 1):
        enc.train(); task_hd.train(); task_adv.train(); probe.train()
        if med_recon is not None: med_recon.train()

        fair      = ep > warmup_ep
        fair_ramp = min(1.0, (ep - warmup_ep) / max(8.0, warmup_ep * 0.4)) if fair else 0.0
        alpha     = global_alpha * fair_ramp if fair else 0.0
        margin    = max(0.005, 0.05 * (1.0 - fair_ramp))

        batch_leakage = []

        for xb_med, xb_task, yb, sb, idx in loader:
            xb_med  = xb_med.to(DEVICE)
            xb_task = xb_task.to(DEVICE)
            yb  = yb.to(DEVICE)
            sb  = sb.to(DEVICE)
            idx = idx.long()

            with torch.no_grad():
                z_task_det, _ = enc(xb_med, xb_task)
            probe_logits = probe(z_task_det)
            probe_loss   = F.cross_entropy(probe_logits, sb)
            opt_probe.zero_grad()
            probe_loss.backward()
            opt_probe.step()

            with torch.no_grad():
                leak_acc = (probe_logits.argmax(1) == sb).float().mean().item()
            batch_leakage.append(leak_acc)

            if use_adv and fair:
                with torch.no_grad():
                    z_task_det2, _ = enc(xb_med, xb_task)
                adv_logits = task_adv(z_task_det2, alpha=alpha)
                adv_loss   = F.cross_entropy(adv_logits, sb)
                opt_task_adv.zero_grad()
                adv_loss.backward()
                nn.utils.clip_grad_norm_(task_adv.parameters(), 1.0)
                opt_task_adv.step()

            z_task, z_med = enc(xb_med, xb_task)
            logit         = task_hd(z_task)
            ybs           = yb * (1 - smooth) + 0.5 * smooth

            loss = F.binary_cross_entropy_with_logits(logit, ybs)

            if fair and fair_ramp > 0:
                if target == 'eo':
                    loss += 1.5 * fw_scale * fair_ramp * fair_loss_eo(logit, yb, sb, margin=margin)
                else:
                    loss += 1.5 * fw_scale * fair_ramp * fair_loss_dp(logit, sb, margin=margin)

            if use_adv and fair and fair_ramp > 0:
                adv_out  = task_adv(z_task, alpha=alpha)
                adv_loss = F.cross_entropy(adv_out, sb)
                loss += 1.0 * fair_ramp * adv_loss

            if fair and fair_ramp > 0 and dynamic_task_adv_w > 0.01:
                z_task_rev = GradRevFn.apply(z_task, alpha * dynamic_task_adv_w * fair_ramp)
                extra_adv  = task_adv(z_task_rev)
                extra_loss = F.cross_entropy(extra_adv, sb)
                loss += 0.5 * dynamic_task_adv_w * fair_ramp * extra_loss

            if fair and fair_ramp > 0:
                loss += 0.6 * fair_ramp * weighted_orthogonality_loss(z_med, z_task, med_weights_t)

            if use_bbn and fair and fair_ramp > 0 and nie_scalar > 1.0:
                z_med_c = z_med  - z_med.mean(0, keepdim=True)
                s_c     = sb.float() - sb.float().mean()
                corr    = torch.mean(z_med_c * s_c.unsqueeze(1), dim=0)
                loss += 0.5 * nie_scalar * fw_scale * fair_ramp * torch.mean(torch.abs(corr))

            if use_bbn and fair and fair_ramp > 0 and med_recon is not None and bbn_tr_tensor is not None:
                med_targets = bbn_tr_tensor[idx].to(DEVICE)
                med_pred    = med_recon(z_med)
                recon_loss  = F.mse_loss(med_pred, med_targets)
                loss += 0.8 * fw_scale * fair_ramp * recon_loss

            if fair and fair_ramp > 0:
                cf_loss = counterfactual_consistency_loss(enc, task_hd, xb_med, xb_task, sb)
                loss += 0.7 * fw_scale * fair_ramp * cf_loss

            opt_main.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
            opt_main.step()

        sched.step()

        if batch_leakage:
            mean_leak = float(np.mean(batch_leakage))
            epoch_leakage.append(mean_leak)
            excess = max(0.0, mean_leak - 0.55)
            dynamic_task_adv_w = float(np.clip(excess * 4.0, 0.0, 1.5))

        if ep == warmup_ep:
            best_state = snap()

        eval_interval = 5 if fair else 10
        if not (ep % eval_interval == 0 or ep == total_ep):
            continue

        pr   = get_proba(enc, task_hd,
                          torch.tensor(X_te_med, dtype=torch.float32),
                          torch.tensor(X_te_task, dtype=torch.float32))
        pred = (pr > 0.5).astype(int)
        acc  = accuracy_score(y_np, pred)
        try:
            eo_v = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_np)))
        except:
            eo_v = 1.0
        try:
            dp_v = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_np)))
        except:
            dp_v = 1.0

        target_metric = eo_v if target == 'eo' else dp_v

        if fair:
            acc_pen   = max(0.0, acc_floor - acc) * 3.0
            penalised = target_metric + acc_pen
            if penalised < best_metric:
                best_metric = penalised
                best_state  = snap()
            if target_metric == 0.0:
                zero_streak += 1
                if zero_streak >= patience:
                    tqdm.write(f"    [{target.upper()}] Early stop ep{ep} (metric=0 x{zero_streak})")
                    break
            else:
                zero_streak = 0

        curve_epochs.append(ep)
        curve_acc.append(acc)
        curve_eo.append(eo_v)
        curve_dp.append(dp_v)

        if (ep % 10 == 0 or ep == total_ep) and ablation_mode == 'full':
            phase    = 'fair' if fair else 'warm'
            leak_str = f"  leak={mean_leak:.3f}" if epoch_leakage else ""
            tqdm.write(f"  [{target.upper()}] ep{ep:>4}: acc={acc:.4f}  "
                       f"EO={eo_v:.4f}  DP={dp_v:.4f}  margin={margin:.4f}  "
                       f"α={alpha:.3f}  dyn_adv={dynamic_task_adv_w:.3f}  [{phase}]{leak_str}")

    if ablation_mode == 'full':
        key = f"{dataset_name}_{target}"
        TRAINING_CURVES[key] = dict(
            epochs=curve_epochs, acc=curve_acc, eo=curve_eo, dp=curve_dp,
            warmup_ep=warmup_ep, leakage=epoch_leakage, target=target)

    load_state(best_state)
    pr_te   = get_proba(enc, task_hd,
                         torch.tensor(X_te_med, dtype=torch.float32),
                         torch.tensor(X_te_task, dtype=torch.float32))
    pred_pre = (pr_te > 0.5).astype(int)
    acc_pre  = accuracy_score(y_np, pred_pre)
    try:
        metric_pre = abs(float(
            equalized_odds_difference(y_np, pred_pre, sensitive_features=s_np)
            if target == 'eo' else
            demographic_parity_difference(y_np, pred_pre, sensitive_features=s_np)))
    except:
        metric_pre = 1.0

    if use_post and metric_pre > 0.03:
        pr_tr = get_proba(enc, task_hd,
                           torch.tensor(X_tr_med, dtype=torch.float32),
                           torch.tensor(X_tr_task, dtype=torch.float32))
        pred_post, metric_post, acc_post = hardt_postprocess(
            pr_te, y_np, s_np, acc_floor, target=target,
            proba_cal_src=(pr_tr, data['y_train']), n_roc_pts=100)
        if metric_post < metric_pre - 0.005 and acc_post >= acc_pre - 0.025:
            return acc_pre, metric_pre, acc_post, metric_post
        else:
            tqdm.write(f"    [Hardt-{target.upper()}] rejected: "
                       f"{metric_pre:.4f}→{metric_post:.4f}, acc {acc_pre:.4f}→{acc_post:.4f}")

    return acc_pre, metric_pre, acc_pre, metric_pre


def train_lcfr(dataset_name, data, bbn_analyzer: BBNPathAnalyzer,
               primary_sens_train, primary_sens_test, baseline_acc,
               baseline_eo: float = None, baseline_dp: float = None,
               ablation_mode: str = 'full'):
    acc_floor = ACC_FLOORS.get(dataset_name, baseline_acc * 0.90)

    if ablation_mode == 'full':
        tqdm.write(f"\n  ─── EO Training (target=equalized_odds) ───")
    set_seed()
    acc_eo_pre, eo_pre, acc_eo_post, eo_post = _train_single(
        dataset_name, data, bbn_analyzer,
        primary_sens_train, primary_sens_test,
        acc_floor, target='eo', ablation_mode=ablation_mode)

    if ablation_mode == 'full':
        tqdm.write(f"\n  ─── DP Training (target=demographic_parity) ───")
    set_seed()
    acc_dp_pre, dp_pre, acc_dp_post, dp_post = _train_single(
        dataset_name, data, bbn_analyzer,
        primary_sens_train, primary_sens_test,
        acc_floor, target='dp', ablation_mode=ablation_mode)

    if ablation_mode == 'full':
        W_DIAG = 72
        print(f"\n{'═'*W_DIAG}")
        print(f"  STAGE-WISE DIAGNOSTIC — {dataset_name.upper()}")
        print(f"{'═'*W_DIAG}")
        print(f"  {'Stage':<28} {'Acc':>7} {'EO':>8} {'DP':>8}  {'ΔAcc':>7} {'ΔEO':>8} {'ΔDP':>8}")
        print(f"  {'-'*72}")
        b_acc = baseline_acc  if baseline_acc  is not None else float('nan')
        b_eo  = baseline_eo   if baseline_eo   is not None else float('nan')
        b_dp  = baseline_dp   if baseline_dp   is not None else float('nan')

        def _fmt(v):
            return f"{v:.4f}" if not (v is None or (isinstance(v, float) and math.isnan(v))) else "  N/A "
        def _delta(cur, ref):
            if ref is None or cur is None: return "   N/A"
            if math.isnan(ref) or math.isnan(cur): return "   N/A"
            return f"{cur - ref:+.4f}"

        print(f"  {'Baseline (MLP)':.<28} {_fmt(b_acc):>7} {_fmt(b_eo):>8} {_fmt(b_dp):>8}  "
              f"{'—':>7} {'—':>8} {'—':>8}")
        print(f"  {'After In-Processing (EO)':.<28} {_fmt(acc_eo_pre):>7} {_fmt(eo_pre):>8} {'—':>8}  "
              f"{_delta(acc_eo_pre,b_acc):>7} {_delta(eo_pre,b_eo):>8} {'—':>8}")
        print(f"  {'After Post-Processing (EO)':.<28} {_fmt(acc_eo_post):>7} {_fmt(eo_post):>8} {'—':>8}  "
              f"{_delta(acc_eo_post,acc_eo_pre):>7} {_delta(eo_post,eo_pre):>8} {'—':>8}")
        print(f"  {'After In-Processing (DP)':.<28} {_fmt(acc_dp_pre):>7} {'—':>8} {_fmt(dp_pre):>8}  "
              f"{_delta(acc_dp_pre,b_acc):>7} {'—':>8} {_delta(dp_pre,b_dp):>8}")
        print(f"  {'After Post-Processing (DP)':.<28} {_fmt(acc_dp_post):>7} {'—':>8} {_fmt(dp_post):>8}  "
              f"{_delta(acc_dp_post,acc_dp_pre):>7} {'—':>8} {_delta(dp_post,dp_pre):>8}")
        print(f"{'═'*W_DIAG}")

        eo_tension = abs(eo_post - dp_post)
        print(f"\n  [Impossibility Observation]")
        print(f"  |EO_post − DP_post| = {eo_tension:.4f}  "
              + ("(tension present)" if eo_tension > 0.01 else "(tension minimal)"))

        STAGE_RECORDS[dataset_name] = dict(
            baseline_acc=b_acc, baseline_eo=b_eo, baseline_dp=b_dp,
            inproc_eo_acc=acc_eo_pre,  inproc_eo=eo_pre,
            post_eo_acc=acc_eo_post,   post_eo=eo_post,
            inproc_dp_acc=acc_dp_pre,  inproc_dp=dp_pre,
            post_dp_acc=acc_dp_post,   post_dp=dp_post,
        )

        secondary_results = {}
        sens_cfg_all = DATASET_CONFIG[dataset_name]["sens_attrs"]
        if len(sens_cfg_all) > 1:
            tqdm.write(f"\n  [Secondary Attribute Fairness Transfer (from EO model)]")
            for s2_tr_key, s2_te_key in sens_cfg_all[1:]:
                if s2_tr_key not in data or s2_te_key not in data:
                    continue
                s2_te = np.asarray(data[s2_te_key])
                attr  = s2_te_key.replace('sens_','').replace('_test','')
                set_seed()
                acc2_pre_eo, eo2_pre, acc2_post_eo, eo2_post = _train_single(
                    dataset_name, data, bbn_analyzer,
                    primary_sens_train, s2_te,
                    acc_floor, target='eo', ablation_mode='no_post')
                set_seed()
                acc2_pre_dp, dp2_pre, acc2_post_dp, dp2_post = _train_single(
                    dataset_name, data, bbn_analyzer,
                    primary_sens_train, s2_te,
                    acc_floor, target='dp', ablation_mode='no_post')
                secondary_results[attr] = dict(
                    acc_pre=acc2_pre_eo,
                    eo_pre=eo2_pre,  dp_pre=dp2_pre,
                    acc_eo_post=acc2_post_eo, eo_post=eo2_post,
                    acc_dp_post=acc2_post_dp, dp_post=dp2_post)
                tqdm.write(f"    attr={attr:<14}  "
                           f"EO  {eo2_pre:.4f} → {eo2_post:.4f}  "
                           f"DP  {dp2_pre:.4f} → {dp2_post:.4f}")

        return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
                acc_dp_pre, dp_pre, acc_dp_post, dp_post,
                secondary_results)

    return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
            acc_dp_pre, dp_pre, acc_dp_post, dp_post, {})


def run_ablation(dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
                 baseline_acc, baseline_eo, baseline_dp):
    tqdm.write(f"\n  [ABLATION STUDY — {dataset_name.upper()}]")
    modes = [
        ('no_bbn',  'No BBN / NIE'),
        ('no_adv',  'No adversary'),
        ('no_post', 'No post-proc'),
        ('full',    'Full LCFR'),
    ]
    abl_res = {}
    for mode, label in modes:
        tqdm.write(f"    Running: {label}")
        set_seed()
        (a_eo_pre, e_pre, a_eo_post, e_post,
         a_dp_pre, d_pre, a_dp_post, d_post, _) = train_lcfr(
            dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
            baseline_acc, baseline_eo=baseline_eo, baseline_dp=baseline_dp,
            ablation_mode=mode)
        abl_res[mode] = dict(
            label=label,
            acc_eo_post=a_eo_post, eo_post=e_post,
            acc_dp_post=a_dp_post, dp_post=d_post,
            acc_eo_pre=a_eo_pre,   eo_pre=e_pre,
            acc_dp_pre=a_dp_pre,   dp_pre=d_pre,
        )
        tqdm.write(f"      EO_post={e_post:.4f}  DP_post={d_post:.4f}  "
                   f"Acc(EO)={a_eo_post:.4f}  Acc(DP)={a_dp_post:.4f}")
    ABLATION_RECORDS[dataset_name] = abl_res
    return abl_res


def run_fairlearn_adversarial(X_tr, y_tr, s_tr, X_te, y_te, s_te):
    in_dim = X_tr.shape[1]
    try:
        predictor = nn.Sequential(nn.Linear(in_dim,64), nn.ReLU(), nn.Linear(64,1), nn.Sigmoid())
        adversary = nn.Sequential(nn.Linear(1,32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid())
        clf = AdversarialFairnessClassifier(
            predictor_model=predictor, adversary_model=adversary,
            epochs=10, batch_size=1024, random_state=SEED)
        clf.fit(X_tr, y_tr, sensitive_features=s_tr)
        pred = clf.predict(X_te)
        acc  = accuracy_score(y_te, pred)
        eo   = r4(equalized_odds_difference(y_te, pred, sensitive_features=s_te))
        dp   = r4(demographic_parity_difference(y_te, pred, sensitive_features=s_te))
        return round(acc, 4), eo, dp
    except Exception as e:
        tqdm.write(f"    [FairLearn] failed: {e}"); return None, None, None


print("=" * 70)
print(f"  LCFR v9 — Split Input | Learned Graph | NIE Gate3 | Task Adversary | Device: {DEVICE}")
print("=" * 70)

LOADERS = {
    "german":    load_german,
    "adult":     load_adult,
    "compas":    load_compas,
    "bank":      load_bank,
    "lawschool": load_lawschool,
    "hospital":  load_hospital,
}

results = {}; fl_results = {}; secondary_all = {}

for idx, (dname, loader_fn) in enumerate(LOADERS.items(), 1):
    print(f"\n{'─'*70}")
    print(f"  [{idx}/{len(LOADERS)}] {dname.upper()}")
    print(f"{'─'*70}")
    set_seed()
    data = loader_fn()

    for split in ('bbn_train_df', 'bbn_test_df'):
        df_fix = data[split]
        for c in df_fix.columns:
            if df_fix[c].dtype == object:
                data[split][c] = df_fix[c].astype('category').cat.codes.astype(int)

    sens_cfg = DATASET_CONFIG[dname]["sens_attrs"][0]
    s_tr_arr = np.asarray(data[sens_cfg[0]])
    s_te_arr = np.asarray(data[sens_cfg[1]])

    sc  = StandardScaler()
    Xb  = sc.fit_transform(data['X_train_nn'])
    Xbt = sc.transform(data['X_test_nn'])
    mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=300, random_state=SEED)
    mlp.fit(Xb, data['y_train'])
    base_pred = mlp.predict(Xbt)
    base_acc  = accuracy_score(data['y_test'], base_pred)
    try:
        base_eo = r4(equalized_odds_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
    except:
        base_eo = float('nan')
    try:
        base_dp = r4(demographic_parity_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
    except:
        base_dp = float('nan')
    tqdm.write(f"\n  [Baseline MLP]  acc={base_acc:.4f}  EO={base_eo:.4f}  DP={base_dp:.4f}")

    bbn_tr_df = data['bbn_train_df'].copy()
    label_col_bbn = 'y'
    if label_col_bbn not in bbn_tr_df.columns:
        possible = [c for c in bbn_tr_df.columns if 'bar' in c or 'recid' in c or 'readmit' in c or 'credit' in c]
        label_col_bbn = possible[0] if possible else bbn_tr_df.columns[-1]

    if label_col_bbn not in bbn_tr_df.columns:
        bbn_tr_df[label_col_bbn] = data['y_train']

    sens_col_bbn = None
    for sc_name in [sens_cfg[0].replace('sens_','').replace('_train',''),
                    sens_cfg[0].replace('_train','')]:
        if sc_name in bbn_tr_df.columns:
            sens_col_bbn = sc_name; break
    if sens_col_bbn is None:
        for col in bbn_tr_df.columns:
            if col not in [label_col_bbn] and bbn_tr_df[col].nunique() <= 3:
                if _compute_mi(bbn_tr_df[col].values.astype(int), s_tr_arr.astype(int)) > 0.05:
                    sens_col_bbn = col; break
    if sens_col_bbn is None:
        sens_col_bbn = bbn_tr_df.columns[0]

    tqdm.write(f"  Building BBN path analyzer with learned graph + NIE gate3 "
               f"(sens_col={sens_col_bbn})...")

    if sens_col_bbn not in bbn_tr_df.columns:
        bbn_tr_df[sens_col_bbn] = s_tr_arr

    analyzer = BBNPathAnalyzer(sens_col=sens_col_bbn, label_col=label_col_bbn)
    analyzer.fit(bbn_tr_df)
    pw  = analyzer.get_pathway_weights()
    nie = analyzer.get_nie_weights()
    vrf = analyzer.get_verification()

    tqdm.write(f"    sens_mi={analyzer.get_sens_mi():.4f}  "
               f"verified_mediators={list(pw.keys())[:5]}")
    tqdm.write(f"    learned_edges (top 5): {analyzer.get_learned_edges()[:5]}")

    top_med = sorted(pw.items(), key=lambda x: x[1], reverse=True)[:8]
    MEDIATOR_SCORES_ALL[dname] = dict(top_med)

    tqdm.write(f"\n  [Mediator Verification (NIE gate3) — top {len(top_med)}]")
    tqdm.write(f"  {'Feature':<22} {'Score':>7} {'NIE':>7} {'MI_SF_n':>8} "
               f"{'MI_FY_n':>8} {'Valid':>6}")
    tqdm.write(f"  {'-'*58}")
    for feat, score in top_med:
        nie_val = nie.get(feat, 0.0)
        v       = vrf.get(feat, {})
        valid   = '  ✓' if v.get('valid', False) else '  ✗'
        tqdm.write(f"  {feat:<22} {score:>7.4f} {nie_val:>7.4f} "
                   f"{v.get('mi_sf_norm',0):>8.4f} {v.get('mi_fy_norm',0):>8.4f} {valid:>6}")

    set_seed()
    (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
     acc_dp_pre, dp_pre, acc_dp_post, dp_post,
     sec_res) = train_lcfr(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc,
                            baseline_eo=base_eo, baseline_dp=base_dp, ablation_mode='full')

    results[dname] = dict(baseline_acc=base_acc,
                          acc_eo_pre=acc_eo_pre, eo_pre=eo_pre,
                          acc_eo_post=acc_eo_post, eo_post=eo_post,
                          acc_dp_pre=acc_dp_pre,  dp_pre=dp_pre,
                          acc_dp_post=acc_dp_post, dp_post=dp_post)
    secondary_all[dname] = sec_res

    run_ablation(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc, base_eo, base_dp)

    tqdm.write("  Running FairLearn Adversarial baseline...")
    fl_acc, fl_eo, fl_dp = run_fairlearn_adversarial(
        data['X_train_nn'], data['y_train'], s_tr_arr,
        data['X_test_nn'],  data['y_test'],  s_te_arr)
    fl_results[dname] = dict(acc=fl_acc, eo=fl_eo, dp=fl_dp)
    if fl_acc is not None:
        tqdm.write(f"  FairLearn: acc={fl_acc:.4f} EO={fl_eo:.4f} DP={fl_dp:.4f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def f2(v):
    if v is None: return ' N/A'
    if isinstance(v, float) and math.isnan(v): return ' N/A'
    return f'{math.floor(abs(float(v))*100)/100:.2f}'

W = 70
print(f"\n{'═'*W}")
print("  FINAL — Equalized Odds (floor-rounded 2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(in)':>8} {'EO(in)':>7} | {'Acc(post)':>9} {'EO(post)':>8}")
print(f"  {'-'*60}")
for d, r in results.items():
    in_flag   = " ✓" if r['eo_pre']  < 0.05 else " !"
    post_flag = " ✓" if r['eo_post'] < 0.05 else " !"
    print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
          f"{f2(r['acc_eo_pre']):>8} {f2(r['eo_pre']):>7}{in_flag} | "
          f"{f2(r['acc_eo_post']):>9} {f2(r['eo_post']):>8}{post_flag}")

print(f"\n{'═'*W}")
print("  FINAL — Demographic Parity (floor-rounded 2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(pre)':>8} {'DP(pre)':>7} | {'Acc(post)':>9} {'DP(post)':>8}")
print(f"  {'-'*60}")
for d, r in results.items():
    flag = " ✓" if r['dp_post'] < 0.05 else " !"
    print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
          f"{f2(r['acc_dp_pre']):>8} {f2(r['dp_pre']):>7} | "
          f"{f2(r['acc_dp_post']):>9} {f2(r['dp_post']):>8}{flag}")

print(f"\n{'═'*W}")
print("  SECONDARY ATTRIBUTE RESULTS")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Attribute':<14} {'EO(pre)':>7} {'EO(post)':>8} {'DP(pre)':>7} {'DP(post)':>8}")
print(f"  {'-'*62}")
for d, sec in secondary_all.items():
    for attr, r2 in sec.items():
        print(f"  {d:<12} {attr:<14} {f2(r2['eo_pre']):>7} {f2(r2['eo_post']):>8} "
              f"{f2(r2['dp_pre']):>7} {f2(r2['dp_post']):>8}")

print(f"\n{'═'*W}")
print("  FAIRLEARN ADVERSARIAL COMPARISON (2 d.p.)")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'LCFR EO':>8} {'LCFR DP':>8} {'LCFR Acc':>9} | "
      f"{'FL Acc':>7} {'FL EO':>6} {'FL DP':>6}")
print(f"  {'-'*65}")
for d in results:
    r = results[d]; fl = fl_results[d]
    print(f"  {d:<12} {f2(r['eo_post']):>8} {f2(r['dp_post']):>8} {f2(r['acc_eo_post']):>9} | "
          f"{f2(fl['acc']):>7} {f2(fl['eo']):>6} {f2(fl['dp']):>6}")
print(f"{'═'*W}")

print(f"\n{'═'*W}")
print("  ABLATION STUDY SUMMARY")
print(f"{'═'*W}")
abl_mode_labels = {
    'no_bbn':  'w/o BBN/NIE',
    'no_adv':  'w/o Adv',
    'no_post': 'w/o Post',
    'full':    'Full LCFR',
}
print(f"  {'Dataset':<12} {'Config':<14} {'EO(in)':>7} {'DP(in)':>7} {'EO(post)':>9} {'DP(post)':>9} {'Acc':>8}")
print(f"  {'-'*70}")
for d in ABLATION_RECORDS:
    for mode, lbl in abl_mode_labels.items():
        ar = ABLATION_RECORDS[d].get(mode, {})
        if not ar: continue
        marker = ' ←' if mode == 'full' else ''
        print(f"  {d:<12} {lbl:<14} {f2(ar.get('eo_pre')):>7} {f2(ar.get('dp_pre')):>7} "
              f"{f2(ar.get('eo_post')):>9} {f2(ar.get('dp_post')):>9} "
              f"{f2(ar.get('acc_eo_pre')):>8}{marker}")
    print(f"  {'-'*70}")
print(f"{'═'*W}")


PLOT_COLOR = {
    'baseline': '#e05c5c', 'preproc': '#e09c3c',
    'inproc':   '#4c8fd4', 'post':    '#3abf6e', 'fl': '#9b6cd4',
}
STAGE_LABELS = ['Baseline', 'In-proc', 'Post-proc']
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})

def _safe(v, default=float('nan')):
    if v is None: return default
    try:
        f = float(v)
        return default if math.isnan(f) else f
    except:
        return default

print("\n  Generating diagnostic plots...")
datasets_ordered = list(results.keys())
n_ds = len(datasets_ordered)

fig1, axes = plt.subplots(2, n_ds, figsize=(3.2*n_ds, 7), constrained_layout=True)
fig1.suptitle("Fairness–Accuracy Tradeoff by Stage", fontsize=13, fontweight='bold', y=1.02)
for col, dname in enumerate(datasets_ordered):
    sr = STAGE_RECORDS.get(dname, {}); r = results[dname]; fl = fl_results[dname]
    eo_vals  = [_safe(sr.get('baseline_eo')), _safe(sr.get('inproc_eo')),   _safe(sr.get('post_eo'))]
    acc_vals = [_safe(sr.get('baseline_acc')), _safe(sr.get('inproc_eo_acc')), _safe(sr.get('post_eo_acc'))]
    dp_vals  = [_safe(sr.get('baseline_dp')), _safe(sr.get('inproc_dp')),   _safe(sr.get('post_dp'))]
    acc_dp_v = [_safe(sr.get('baseline_acc')), _safe(sr.get('inproc_dp_acc')), _safe(sr.get('post_dp_acc'))]
    colors = [PLOT_COLOR['baseline'], PLOT_COLOR['inproc'], PLOT_COLOR['post']]
    x = np.arange(len(STAGE_LABELS))
    ax_eo = axes[0, col]; ax_acc = ax_eo.twinx()
    ax_eo.bar(x, eo_vals, color=colors, alpha=0.82, width=0.5, zorder=3)
    ax_acc.plot(x, acc_vals, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
    if fl.get('acc') is not None:
        ax_eo.axhline(y=_safe(fl.get('eo')), color=PLOT_COLOR['fl'], linestyle=':', linewidth=1.5)
    ax_eo.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.6)
    ax_eo.set_title(f"{dname.upper()}\nEO by Stage", fontsize=9, fontweight='bold')
    ax_eo.set_xticks(x); ax_eo.set_xticklabels(STAGE_LABELS, rotation=30, ha='right', fontsize=7.5)
    ax_eo.set_ylabel("EO", fontsize=8); ax_acc.set_ylabel("Acc", fontsize=8)
    ax_eo.set_ylim(bottom=0)
    ax_dp = axes[1, col]; ax_da = ax_dp.twinx()
    ax_dp.bar(x, dp_vals, color=colors, alpha=0.82, width=0.5, zorder=3)
    ax_da.plot(x, acc_dp_v, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
    if fl.get('acc') is not None:
        ax_dp.axhline(y=_safe(fl.get('dp')), color=PLOT_COLOR['fl'], linestyle=':', linewidth=1.5)
    ax_dp.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.6)
    ax_dp.set_title("DP by Stage", fontsize=9)
    ax_dp.set_xticks(x); ax_dp.set_xticklabels(STAGE_LABELS, rotation=30, ha='right', fontsize=7.5)
    ax_dp.set_ylabel("DP", fontsize=8); ax_da.set_ylabel("Acc", fontsize=8)
    ax_dp.set_ylim(bottom=0)
legend_patches = [mpatches.Patch(color=PLOT_COLOR[k], label=l)
                  for k, l in [('baseline','Baseline'),('inproc','In-proc'),
                                ('post','Post-proc'),('fl','FairLearn')]]
fig1.legend(handles=legend_patches, loc='lower center', ncol=4, bbox_to_anchor=(0.5,-0.04), fontsize=8)
fig1.savefig(os.path.join(RESULTS_DIR, 'plot1_fairness_accuracy_tradeoff.png'), dpi=150, bbox_inches='tight')
plt.close(fig1); print("  Saved: plot1_fairness_accuracy_tradeoff.png")

fig2, axes2 = plt.subplots(2, n_ds, figsize=(3.5*n_ds, 7.0), constrained_layout=True)
if n_ds == 1: axes2 = axes2.reshape(2, 1)
fig2.suptitle("Training Curves — EO model (top) / DP model (bottom)", fontsize=12, fontweight='bold')
for col, dname in enumerate(datasets_ordered):
    for row, tgt in enumerate(['eo', 'dp']):
        tc = TRAINING_CURVES.get(f"{dname}_{tgt}", {})
        if not tc: continue
        ax = axes2[row, col]; ax2 = ax.twinx()
        color = PLOT_COLOR['inproc'] if tgt == 'eo' else PLOT_COLOR['baseline']
        ax.plot(tc['epochs'], tc[tgt], color=color, linewidth=1.4, label=tgt.upper())
        ax2.plot(tc['epochs'], tc['acc'], color='#333', linewidth=1.2, linestyle='--', label='Acc')
        ax.axvline(x=tc['warmup_ep'], color='#aaa', linestyle=':', linewidth=1.1)
        ax.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.set_title(f"{dname.upper()} — {tgt.upper()} model", fontsize=9, fontweight='bold')
        ax.set_xlabel("Epoch", fontsize=8); ax.set_ylabel(tgt.upper(), fontsize=8)
        ax2.set_ylabel("Accuracy", fontsize=8); ax.set_ylim(bottom=0)
        lines1, labs1 = ax.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax.legend(lines1+lines2, labs1+labs2, fontsize=7, loc='upper right')
fig2.savefig(os.path.join(RESULTS_DIR, 'plot2_training_curves.png'), dpi=150, bbox_inches='tight')
plt.close(fig2); print("  Saved: plot2_training_curves.png")

n_med_plots = sum(1 for d in datasets_ordered if MEDIATOR_SCORES_ALL.get(d))
if n_med_plots > 0:
    fig3, axes3 = plt.subplots(1, n_med_plots, figsize=(3.8*n_med_plots, 4.5), constrained_layout=True)
    if n_med_plots == 1: axes3 = [axes3]
    fig3.suptitle("Mediator Bias Scores (NIE-gate3 verified)", fontsize=12, fontweight='bold')
    plot_col = 0
    for dname in datasets_ordered:
        ms = MEDIATOR_SCORES_ALL.get(dname)
        if not ms: continue
        ax = axes3[plot_col]; plot_col += 1
        feats  = list(ms.keys())[:8]
        scores = [ms[f] for f in feats]
        vrf_d  = {}
        cmap   = plt.cm.YlOrRd
        bar_colors = [cmap(0.35 + 0.6*s) for s in scores]
        bars = ax.barh(feats[::-1], scores[::-1], color=bar_colors[::-1],
                       edgecolor='white', height=0.55, zorder=3)
        for bar, sc in zip(bars, scores[::-1]):
            ax.text(sc+0.01, bar.get_y()+bar.get_height()/2,
                    f"{sc:.3f}", va='center', fontsize=7.5)
        ax.axvline(x=0.3, color='#999', linestyle='--', linewidth=0.9, alpha=0.7)
        ax.set_xlim(0, 1.15)
        ax.set_title(f"{dname.upper()}", fontsize=9, fontweight='bold')
        ax.set_xlabel("Normalised Score", fontsize=8)
        ax.tick_params(axis='y', labelsize=8)
    fig3.savefig(os.path.join(RESULTS_DIR, 'plot3_mediator_bias_scores.png'), dpi=150, bbox_inches='tight')
    plt.close(fig3); print("  Saved: plot3_mediator_bias_scores.png")

has_secondary = any(bool(v) for v in secondary_all.values())
if has_secondary:
    sec_rows = [(d, attr, r2) for d, sec in secondary_all.items() for attr, r2 in sec.items()]
    n_sec = len(sec_rows)
    fig4, axes4 = plt.subplots(1, max(n_sec,1), figsize=(4.0*max(n_sec,1), 4.5), constrained_layout=True)
    if n_sec == 1: axes4 = [axes4]
    fig4.suptitle("Fairness Transfer — Secondary Attribute", fontsize=12, fontweight='bold')
    for col, (d, attr, r2) in enumerate(sec_rows):
        ax = axes4[col]
        x = np.arange(2); w = 0.32
        pre_v  = [_safe(r2.get('eo_pre')),  _safe(r2.get('dp_pre'))]
        post_v = [_safe(r2.get('eo_post')), _safe(r2.get('dp_post'))]
        b1 = ax.bar(x-w/2, pre_v,  width=w, color=PLOT_COLOR['baseline'], alpha=0.85, label='Pre',  zorder=3)
        b2 = ax.bar(x+w/2, post_v, width=w, color=PLOT_COLOR['post'],     alpha=0.85, label='Post', zorder=3)
        for bar in list(b1)+list(b2):
            h = bar.get_height()
            if not math.isnan(h):
                ax.text(bar.get_x()+bar.get_width()/2, h+0.005, f"{h:.3f}",
                        ha='center', va='bottom', fontsize=7.5)
        ax.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.7)
        ax.set_xticks(x); ax.set_xticklabels(['EO','DP']); ax.set_ylim(bottom=0)
        ax.set_title(f"{d.upper()}\n{attr}", fontsize=9, fontweight='bold')
        ax.legend(fontsize=7)
    fig4.savefig(os.path.join(RESULTS_DIR, 'plot4_fairness_transfer.png'), dpi=150, bbox_inches='tight')
    plt.close(fig4); print("  Saved: plot4_fairness_transfer.png")

fig5, axes5 = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
fig5.suptitle("LCFR v9 vs FairLearn Adversarial", fontsize=12, fontweight='bold')
x = np.arange(n_ds); w = 0.22; names = [d.upper() for d in datasets_ordered]
for ax_i, (mk_lcfr, mk_fl, title) in enumerate([('eo_post','eo','Equalized Odds'),
                                                  ('dp_post','dp','Demographic Parity')]):
    ax = axes5[ax_i]
    lcfr_v = [_safe(results[d].get(mk_lcfr)) for d in datasets_ordered]
    fl_v   = [_safe(fl_results[d].get(mk_fl)) for d in datasets_ordered]
    acc_lc = [_safe(results[d].get('acc_eo_post' if mk_lcfr=='eo_post' else 'acc_dp_post'))
              for d in datasets_ordered]
    acc_fl = [_safe(fl_results[d].get('acc')) for d in datasets_ordered]
    b1 = ax.bar(x-w*1.5, lcfr_v, width=w, color=PLOT_COLOR['post'],  alpha=0.88, label='LCFR', zorder=3)
    b2 = ax.bar(x-w*0.5, fl_v,   width=w, color=PLOT_COLOR['fl'],    alpha=0.88, label='FL',   zorder=3)
    ax2 = ax.twinx()
    ax2.plot(x+w*0.5, acc_lc, 'D--', color=PLOT_COLOR['post'], linewidth=1.4, markersize=6, label='LCFR Acc')
    ax2.plot(x+w*1.5, acc_fl, 's--', color=PLOT_COLOR['fl'],   linewidth=1.4, markersize=6, label='FL Acc')
    for bar in list(b1)+list(b2):
        h = bar.get_height()
        if not math.isnan(h):
            ax.text(bar.get_x()+bar.get_width()/2, h+0.002, f"{h:.3f}",
                    ha='center', va='bottom', fontsize=7, rotation=60)
    ax.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
    ax.set_ylabel(title, fontsize=9); ax2.set_ylabel("Accuracy", fontsize=9)
    ax.set_ylim(bottom=0); ax.set_title(title, fontsize=10, fontweight='bold')
    l1, n1 = ax.get_legend_handles_labels(); l2, n2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, n1+n2, fontsize=7, loc='upper right')
fig5.savefig(os.path.join(RESULTS_DIR, 'plot5_lcfr_vs_fairlearn.png'), dpi=150, bbox_inches='tight')
plt.close(fig5); print("  Saved: plot5_lcfr_vs_fairlearn.png")

fig6, axes6 = plt.subplots(1, n_ds, figsize=(3.5*n_ds, 4.0), constrained_layout=True)
if n_ds == 1: axes6 = [axes6]
fig6.suptitle("EO vs DP Tension (separate models)", fontsize=12, fontweight='bold')
for col, dname in enumerate(datasets_ordered):
    ax = axes6[col]; sr = STAGE_RECORDS.get(dname, {}); fl = fl_results[dname]
    stages = [
        (_safe(sr.get('baseline_eo')), _safe(sr.get('baseline_dp')), 'Baseline',  PLOT_COLOR['baseline']),
        (_safe(sr.get('inproc_eo')),   _safe(sr.get('inproc_dp')),   'In-proc',   PLOT_COLOR['inproc']),
        (_safe(sr.get('post_eo')),     _safe(sr.get('post_dp')),     'Post-proc', PLOT_COLOR['post']),
    ]
    if fl.get('acc') is not None:
        stages.append((_safe(fl.get('eo')), _safe(fl.get('dp')), 'FairLearn', PLOT_COLOR['fl']))
    for eo_v, dp_v, label, color in stages:
        if not math.isnan(eo_v) and not math.isnan(dp_v):
            ax.scatter(eo_v, dp_v, color=color, s=90, zorder=4, edgecolors='white', linewidths=0.8)
            ax.annotate(label, (eo_v, dp_v), textcoords='offset points', xytext=(5,4),
                        fontsize=7, color=color)
    xs = [s[0] for s in stages if not math.isnan(s[0])]
    ys = [s[1] for s in stages if not math.isnan(s[1])]
    if xs and ys:
        mx = max(max(xs), max(ys), 0.05) * 1.2
        t  = np.linspace(0, mx, 50)
        ax.plot(t, t, color='#bbb', linestyle='--', linewidth=0.9, alpha=0.6)
    ax.axvline(x=0.05, color='#cc2222', linestyle=':', linewidth=0.9, alpha=0.6)
    ax.axhline(y=0.05, color='#cc2222', linestyle=':', linewidth=0.9, alpha=0.6)
    ax.set_xlabel("EO (from EO model)", fontsize=8); ax.set_ylabel("DP (from DP model)", fontsize=8)
    ax.set_title(f"{dname.upper()}\nEO–DP", fontsize=9, fontweight='bold')
    ax.set_xlim(left=0); ax.set_ylim(bottom=0)
fig6.savefig(os.path.join(RESULTS_DIR, 'plot6_eo_dp_tension.png'), dpi=150, bbox_inches='tight')
plt.close(fig6); print("  Saved: plot6_eo_dp_tension.png")

if ABLATION_RECORDS:
    abl_modes = ['no_bbn', 'no_adv', 'no_post', 'full']
    abl_short  = ['w/o BBN', 'w/o Adv', 'w/o Post', 'Full']
    n_abl = len(abl_modes)
    fig7, axes7 = plt.subplots(2, n_ds, figsize=(3.5*n_ds, 7), constrained_layout=True)
    fig7.suptitle("Ablation Study — EO & DP by Component", fontsize=12, fontweight='bold')
    for col, dname in enumerate(datasets_ordered):
        abl = ABLATION_RECORDS.get(dname, {})
        eo_vals  = [_safe(abl.get(m, {}).get('eo_post'))     for m in abl_modes]
        dp_vals  = [_safe(abl.get(m, {}).get('dp_post'))     for m in abl_modes]
        acc_eo_v = [_safe(abl.get(m, {}).get('acc_eo_post')) for m in abl_modes]
        acc_dp_v = [_safe(abl.get(m, {}).get('acc_dp_post')) for m in abl_modes]
        bar_colors = [PLOT_COLOR['baseline'], PLOT_COLOR['inproc'],
                      PLOT_COLOR['preproc'],  PLOT_COLOR['post']]
        x_abl = np.arange(n_abl)
        ax_eo = axes7[0, col]; ax_ea = ax_eo.twinx()
        ax_eo.bar(x_abl, eo_vals, color=bar_colors, alpha=0.85, width=0.55, zorder=3)
        ax_ea.plot(x_abl, acc_eo_v, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
        ax_eo.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.7)
        ax_eo.set_title(f"{dname.upper()}\nEO — Ablation", fontsize=9, fontweight='bold')
        ax_eo.set_xticks(x_abl); ax_eo.set_xticklabels(abl_short, rotation=30, ha='right', fontsize=7.5)
        ax_eo.set_ylabel("EO", fontsize=8); ax_ea.set_ylabel("Acc", fontsize=8); ax_eo.set_ylim(bottom=0)
        ax_dp = axes7[1, col]; ax_da = ax_dp.twinx()
        ax_dp.bar(x_abl, dp_vals, color=bar_colors, alpha=0.85, width=0.55, zorder=3)
        ax_da.plot(x_abl, acc_dp_v, 'o--', color='#222', linewidth=1.4, markersize=5, zorder=4)
        ax_dp.axhline(y=0.05, color='#cc2222', linestyle='--', linewidth=0.9, alpha=0.7)
        ax_dp.set_title("DP — Ablation", fontsize=9)
        ax_dp.set_xticks(x_abl); ax_dp.set_xticklabels(abl_short, rotation=30, ha='right', fontsize=7.5)
        ax_dp.set_ylabel("DP", fontsize=8); ax_da.set_ylabel("Acc", fontsize=8); ax_dp.set_ylim(bottom=0)
    fig7.savefig(os.path.join(RESULTS_DIR, 'plot7_ablation_study.png'), dpi=150, bbox_inches='tight')
    plt.close(fig7); print("  Saved: plot7_ablation_study.png")

print(f"\n{'═'*W}")
print("  LIMITATIONS / FAILURE CASE NOTES")
print(f"{'═'*W}")
for dname in datasets_ordered:
    r  = results[dname]; sr = STAGE_RECORDS.get(dname, {})
    b_acc = _safe(sr.get('baseline_acc')); p_acc = _safe(sr.get('post_eo_acc'))
    eo_v  = _safe(r.get('eo_post'));       dp_v  = _safe(r.get('dp_post'))
    acc_drop = b_acc - p_acc if not (math.isnan(b_acc) or math.isnan(p_acc)) else float('nan')
    notes = []
    if not math.isnan(acc_drop) and acc_drop > 0.03:
        notes.append(f"accuracy cost {acc_drop:.3f} > 0.03")
    if not math.isnan(eo_v) and eo_v > 0.05:
        notes.append(f"EO={eo_v:.4f} above target")
    if not math.isnan(dp_v) and dp_v > 0.05:
        notes.append(f"DP={dp_v:.4f} above target")
    if not notes:
        notes.append("no critical failure detected")
    print(f"  {dname.upper():<12}  {' | '.join(notes)}")

print(f"\n  Architecture summary (v9):")
print(f"  · Preprocessing filters label leaks + sensitive proxies before BBN analysis.")
print(f"  · BBN graph structure learned via HillClimbSearch + BIC (lightweight).")
print(f"  · Gate3 = NIE-based: nie_ratio = NIE(S→M→Y) / MI(S,Y) > 0.01.")
print(f"  · Thresholds are relative (normalised MI): MI_SF/max(H_S,H_F) > 0.02.")
print(f"  · SplitInputEncoder: x_med → z_med branch, x_task → z_task branch (no shared layer).")
print(f"  · Adversary operates on z_task (not z_med) via gradient reversal.")
print(f"  · Mediator recon weight=0.8, counterfactual loss weight=0.7 (both increased).")
print(f"  · Orthogonality weighted per mediator by BBN pathway scores.")
print(f"  · EO and DP are completely independent _train_single calls.")
print(f"{'═'*W}")
print(f"\n  All plots saved to: {RESULTS_DIR}")

  LCFR v9 — Split Input | Learned Graph | NIE Gate3 | Task Adversary | Device: cuda

──────────────────────────────────────────────────────────────────────
  [1/6] GERMAN
──────────────────────────────────────────────────────────────────────

  [Baseline MLP]  acc=0.7050  EO=0.1573  DP=0.1088
  Building BBN path analyzer with learned graph + NIE gate3 (sens_col=status)...


  0%|          | 0/200 [00:00<?, ?it/s]

    sens_mi=0.0661  verified_mediators=['credit_history']
    learned_edges (top 5): [('status', 'y')]

  [Mediator Verification (NIE gate3) — top 1]
  Feature                  Score     NIE  MI_SF_n  MI_FY_n  Valid
  ----------------------------------------------------------
  credit_history          1.0000  1.0000   0.0275   0.0243      ✓

  ─── EO Training (target=equalized_odds) ───
  [EO] ep  10: acc=0.7050  EO=0.0417  DP=0.0174  margin=0.0500  α=0.000  dyn_adv=0.995  [warm]  leak=0.799
  [EO] ep  20: acc=0.7300  EO=0.2292  DP=0.1545  margin=0.0500  α=0.000  dyn_adv=1.035  [warm]  leak=0.809
  [EO] ep  30: acc=0.7450  EO=0.0282  DP=0.0357  margin=0.0250  α=0.282  dyn_adv=1.115  [fair]  leak=0.829
  [EO] ep  40: acc=0.7100  EO=0.2917  DP=0.1013  margin=0.0050  α=0.564  dyn_adv=1.115  [fair]  leak=0.829
  [EO] ep  50: acc=0.7350  EO=0.2917  DP=0.0889  margin=0.0050  α=0.564  dyn_adv=1.160  [fair]  leak=0.840
  [EO] ep  60: acc=0.7300  EO=0.3958  DP=0.1362  margin=0.0050  α=0.564  dy

  0%|          | 0/200 [00:00<?, ?it/s]

    sens_mi=0.0579  verified_mediators=['marital-status', 'relationship', 'education', 'hours-per-week']
    learned_edges (top 5): [('age', 'hours-per-week'), ('marital-status', 'y')]

  [Mediator Verification (NIE gate3) — top 4]
  Feature                  Score     NIE  MI_SF_n  MI_FY_n  Valid
  ----------------------------------------------------------
  marital-status          1.0000  1.0000   0.1274   0.0856      ✓
  relationship            0.7942  0.7826   0.0950   0.0767      ✓
  education               0.1961  0.1505   0.0204   0.0313      ✓
  hours-per-week          0.1796  0.0814   0.0294   0.0301      ✓

  ─── EO Training (target=equalized_odds) ───
  [EO] ep  10: acc=0.8452  EO=0.0875  DP=0.1899  margin=0.0500  α=0.000  dyn_adv=0.611  [warm]  leak=0.703
  [EO] ep  20: acc=0.8482  EO=0.0692  DP=0.1672  margin=0.0500  α=0.000  dyn_adv=0.727  [warm]  leak=0.732
  [EO] ep  30: acc=0.8039  EO=0.1672  DP=0.0950  margin=0.0050  α=0.531  dyn_adv=0.194  [fair]  leak=0.598
  [EO] ep

  0%|          | 0/200 [00:00<?, ?it/s]

    sens_mi=0.0014  verified_mediators=['c_charge_desc']
    learned_edges (top 5): [('age', 'c_charge_desc')]

  [Mediator Verification (NIE gate3) — top 1]
  Feature                  Score     NIE  MI_SF_n  MI_FY_n  Valid
  ----------------------------------------------------------
  c_charge_desc           1.0000  1.0000   0.0510   0.0432      ✓

  ─── EO Training (target=equalized_odds) ───
  [EO] ep  10: acc=0.6826  EO=0.3225  DP=0.3043  margin=0.0500  α=0.000  dyn_adv=0.266  [warm]  leak=0.616
  [EO] ep  20: acc=0.6907  EO=0.3166  DP=0.3001  margin=0.0500  α=0.000  dyn_adv=0.731  [warm]  leak=0.733
  [EO] ep  30: acc=0.6696  EO=0.0779  DP=0.1022  margin=0.0050  α=0.306  dyn_adv=0.825  [fair]  leak=0.756
  [EO] ep  40: acc=0.6753  EO=0.1002  DP=0.1179  margin=0.0050  α=0.306  dyn_adv=0.627  [fair]  leak=0.707
  [EO] ep  50: acc=0.6696  EO=0.0987  DP=0.1259  margin=0.0050  α=0.306  dyn_adv=0.433  [fair]  leak=0.658
  [EO] ep  60: acc=0.6729  EO=0.0972  DP=0.1195  margin=0.0050  α=0

  0%|          | 0/200 [00:00<?, ?it/s]

    sens_mi=0.0063  verified_mediators=['housing']
    learned_edges (top 5): [('housing', 'y')]

  [Mediator Verification (NIE gate3) — top 1]
  Feature                  Score     NIE  MI_SF_n  MI_FY_n  Valid
  ----------------------------------------------------------
  housing                 1.0000  1.0000   0.0217   0.0794      ✓

  ─── EO Training (target=equalized_odds) ───
  [EO] ep  10: acc=0.8317  EO=0.0112  DP=0.0082  margin=0.0500  α=0.000  dyn_adv=0.145  [warm]  leak=0.586
  [EO] ep  20: acc=0.8368  EO=0.0276  DP=0.0044  margin=0.0188  α=0.203  dyn_adv=0.184  [fair]  leak=0.596
  [EO] ep  30: acc=0.8381  EO=0.0105  DP=0.0003  margin=0.0050  α=0.325  dyn_adv=0.000  [fair]  leak=0.522
  [EO] ep  40: acc=0.8470  EO=0.0074  DP=0.0122  margin=0.0050  α=0.325  dyn_adv=0.130  [fair]  leak=0.582
  [EO] ep  50: acc=0.8458  EO=0.0226  DP=0.0074  margin=0.0050  α=0.325  dyn_adv=0.105  [fair]  leak=0.576
  [EO] ep  60: acc=0.8464  EO=0.0183  DP=0.0149  margin=0.0050  α=0.325  dyn_adv=